## Running the Analysis

The Jupyter Notebook is provided with its original code and analytical explanations. To reproduce the analysis and generate the visualizations:

1. Clone or download this repository.
2. Install the required Python dependencies.
3. Download the original Olist Brazilian E-commerce dataset from its source.
4. Configure the dataset paths in the notebook to match your local environment.
5. Open the notebook in Jupyter Notebook or JupyterLab.
6. Execute the cells sequentially to reproduce the analysis, statistical results, and visualizations.

**Note:** Notebook outputs have been cleared to keep the repository lightweight. Interactive Plotly charts and Folium maps are provided separately where available.

Import Necessary Packages

In [ ]:
# %pip install plotly
# %pip install dash
# %pip install geopandas
# %pip install folium
# %pip install ipywidgets
# %pip install pypalettes
# %pip install drawarrow
# %pip install pyfonts
# %pip install highlight-text
# %pip install statsmodels
# %pip install kaleido

# Import Necessary Libraries:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import json
from urllib.request import urlopen
import geopandas as gpd
import requests
import folium
from folium.plugins import MarkerCluster
from ipywidgets import interact
import branca.colormap as cm
import branca
from statsmodels.tsa.statespace.sarimax import SARIMAX

Load and preview the data

In [ ]:
#Load Data
orders = pd.read_csv('datasets/olist_orders_dataset.csv')
order_items = pd.read_csv('datasets/olist_order_items_dataset.csv')
customers = pd.read_csv('datasets/olist_customers_dataset.csv')
products = pd.read_csv('datasets/olist_products_dataset.csv')
sellers = pd.read_csv('datasets/olist_sellers_dataset.csv')
reviews = pd.read_csv('datasets/olist_order_reviews_dataset.csv')
payments = pd.read_csv('datasets/olist_order_payments_dataset.csv')
geo_location = pd.read_csv('datasets/olist_geolocation_dataset.csv')
products_name = pd.read_csv('datasets/product_category_name_translation.csv')

#Preview the Data
products_name.head()

Inspect the loaded data for missing values and duplicates

In [ ]:
print("📦 orders.csv")
print(orders.shape)
print(orders.dtypes)
print(orders.columns)
print(orders.isnull().sum())
print(orders.duplicated().sum())
print('\n')
print("📦 order_items.csv")
print(order_items.shape)
print(order_items.dtypes)
print(order_items.columns)
print(order_items.isnull().sum())
print(order_items.duplicated().sum())
print('\n')
print("📦 customers.csv")
print(customers.shape)
print(customers.dtypes)
print(customers.columns)
print(customers.isnull().sum())
print(customers.duplicated().sum())
print('\n')
print("📦 products.csv")
print(products.shape)
print(products.dtypes)
print(products.columns)
print(products.isnull().sum())
print(products.duplicated().sum())
print('\n')
print("📦 sellers.csv")
print(sellers.shape)
print(sellers.dtypes)
print(sellers.columns)
print(sellers.isnull().sum())
print(sellers.duplicated().sum())
print('\n')
print("📦 payments.csv")
print(payments.shape)
print(payments.dtypes)
print(payments.columns)
print(payments.isnull().sum())
print(payments.duplicated().sum())
print('\n')
print("📦 reviews.csv")
print(reviews.shape)
print(reviews.dtypes)
print(reviews.columns)
print(reviews.isnull().sum())
print(reviews.duplicated().sum())



# Data Cleaning Strategy:

**Missing Values**     | Identified missing columns in `orders`, `products`, `reviews` dataset plan to imputate or remove.

**Data Types**         | Convert datetime fields to datetime objects

<h3>For orders dataset,</h3>
<li>columns such as 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date' has missing values.
    
<li>Droppping the rows for some missing values will not be valid here by dropping we may lose some important information.

<li>And as the missing values are from datevalues it would be only appropriate to fill them with unknown or 'NaT'.

<li>Decision: Fill all nulls with NaT → preserves timeline while being semantically accurate.</li>
    
<h3>For products dataset,</h3>
<li>columns such as 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm' has missing values.
    
<li>We can drop all the missing value rows where the main detail column such as product_category_name are missing but the other details such as product_weight_g, product_length_cm, product_height_cm, product_width_cm are present so the dropping depends on necessity of these columns for our analysis. the other take would be creting a copy of the dataset where the product category name is present and fill the other missing details with their mean values.</li>

<h3>For reviews dataset</h3>
<li>columns such as 'review_comment_title', 'review_comment_message' has missing values.
    
<li>I think dropping rows will not be relevant here as it may drop the review_score which may be relevant if we consider reviews for order_id's in our analysis, instead we can fill them with 'Unknown'.
    
<li>Decision: Fill both text fields with 'Unknown' → preserves review score + timestamps</li>


In [ ]:
# Inspect the null-containing columns and either drop it, fill it or plan further
# For orders table
orders[orders.isnull().any(axis=1)]
# Fill NaTs in orders
orders[['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']] = \
    orders[['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']].fillna(pd.NaT)


# For products table
products[products.isnull().any(axis=1)]
# Creating a copy of product data with product category name present
products_with_category = products[products["product_category_name"].notnull()]

# Fill only for rows where category exists but physical values are missing
cols_to_fill = ['product_weight_g', 'product_length_cm', 
                'product_height_cm', 'product_width_cm']

products_with_category[cols_to_fill] = products_with_category[cols_to_fill].fillna(products_with_category[cols_to_fill].mean())


# For reviews table
reviews[reviews.isnull().any(axis=1)]
# Fill missing review fields with 'Unknown'
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('Unknown')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('Unknown')



<h3>Converting the following columns to it's correct and meaningful datatypes</h3>

In [ ]:
# For orders:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# For order_items:
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

# For reviews:
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

<h3>Summary of order lifecycle date</h3>

In [ ]:
orders[['order_purchase_timestamp', 
        'order_approved_at', 
        'order_delivered_carrier_date', 
        'order_delivered_customer_date', 
        'order_estimated_delivery_date']].describe()

<h3>Summary Table</h3>

| Timestamp                       | Insight                                                                                         |
| ------------------------------- | ----------------------------------------------------------------------------------------------- |
| **Earliest order**              | `2016-09-04`                                                                                    |
| **Latest order**                | `2018-10-17`                                                                                    |
| **Most active ordering period** | Between **Jan 2018 to Aug 2018** (50%+ orders)                                                  |
| **Purchase → Approval gap**     | Mean approval happens \~**10 hours** after purchase                                             |
| **Shipping delays (carrier)**   | On average, shipped \~**4 days** after purchase                                                 |
| **Customer delivery time**      | Delivered \~**14 days** after purchase (avg)                                                    |
| **Estimated delivery**          | \~**10 days after actual delivery** → Could be early delivery insights                          |
| **Missing timestamps**          | Gaps in approval, shipping, and delivery dates could imply returns, preorders, or cancellations |

<h3>Invalid records</h3>
<li>If you look at the table below, the 'order_delivered_carrier_date'(shipping date) is greater than the 'order_purchase_timestamp'(purchase date) "how could the shipping date be higher than then the purchase date?"
    
<li>Orders should never be shipped before being placed — those 166 rows with negative shipping_days indicate:
<ul><li>Maybe system recording bugs or bad data entries.</ul>
        
<li>We need to clean those invalid records before further analysis, that's why i took the average(mean) between the purchase date and the 'order_delivered_customer_date' (customer delivery date) and replaced it for the invalid records.</li>

In [ ]:
orders[orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]]

In [ ]:
orders["order_delivered_carrier_date"] = np.where((orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]),
                                                  orders["order_purchase_timestamp"] + ((orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]) / 2),
                                                  orders["order_delivered_carrier_date"])


<h3>Key Feauture Engineering</h3>
We’ll now create some critical date-based columns to prep for deeper EDA:

<li>delivery_days = days from purchase to delivery

<li>delay_vs_estimate = delivery - estimated

<li>shipping_days = days between purchase → carrier delivery

<li>order_weekday, order_month, order_hour = for trend analysis</li>

In [ ]:
def get_time_of_day(hour):
    if hour < 5:
        return "Mid-night"
    elif hour < 12:
        return "Morning"
    elif hour < 17:
        return "Afternoon"
    elif hour < 19:
        return "Evening"
    else:
        return "Night"
    

orders = orders.assign(delivery_days = lambda x : (x["order_delivered_customer_date"] - x["order_purchase_timestamp"]).dt.days,
                      delay_vs_estimate = lambda x : (x["order_estimated_delivery_date"] - x["order_delivered_customer_date"]).dt.days,
                      shipping_days = lambda x : (x["order_delivered_carrier_date"] - x["order_purchase_timestamp"]).dt.days,
                      order_year = lambda x : x["order_purchase_timestamp"].dt.year,
                      order_quarter = lambda x : x['order_purchase_timestamp'].dt.to_period('Q').astype(str),
                      order_month = lambda x : x["order_purchase_timestamp"].dt.month,
                      order_month_name = lambda x : x["order_purchase_timestamp"].dt.strftime('%b'),
                      order_year_month = lambda x : x["order_purchase_timestamp"].dt.strftime('%Y-%m'),
                      order_day = lambda x : x["order_purchase_timestamp"].dt.day,
                      order_day_name = lambda x: x["order_purchase_timestamp"].dt.strftime('%a'),
                      order_hour = lambda x : x["order_purchase_timestamp"].dt.hour,
                      order_time_day = lambda x : x["order_purchase_timestamp"].dt.hour.apply(get_time_of_day))

orders
                                                   

# Exploratary Data Analysis(EDA):

# Order Performance:
To understand demand patterns, identify seasonal peaks, detect growth signals, and uncover operational timing insights useful for advertising, inventory planning, and staffing.

Orders by Year and Month, Day of the Week and Time of the Day. 
 
<li>During the evolution of total orders over the years is there any seasonal trends on Brazilian E-commerce?
<li>Are customers more active on weekends or weekdays?
<li>What time of day do customers place orders the most?</li>
    

In [ ]:
yearly_month_orders = orders.groupby('order_year_month')['order_id'].count().reset_index().rename(columns={'order_id' : 'total_orders', 'order_year_month' : 'year_month'})

weekday_orders = orders.groupby('order_day_name').size().reset_index(name='count')
weekday_orders['percentage'] = weekday_orders['count'].transform(lambda x: ((x/x.sum()) * 100).round(2))

timely_orders = orders.groupby('order_time_day').size().reset_index(name='count')
timely_orders['percentage'] = timely_orders['count'].transform(lambda x : ((x/x.sum()) * 100).round(2))

fig = make_subplots(rows=2, cols=2, row_heights=[0.6, 0.4],
                    specs=[[{'type':'xy', 'colspan':2}, None],
                           [{'type':'xy'}, {'type':'xy'}]],
                    subplot_titles=['Evolution of Total Orders over the Years', 'Order Distribution by Weekday', 'Order Distribution by Time of Day'],
                    vertical_spacing = 0.20)

# Evolution of Total Orders over the Years (lineplot)
fig.add_trace(go.Scatter(
    x = yearly_month_orders['year_month'],
    y = yearly_month_orders['total_orders'],
    mode = 'lines+markers+text',
    name = 'Total Orders',
    text = [f'{c:,}' for c in yearly_month_orders["total_orders"]],
    line=dict(
            color='#8d4ecb'),
    hovertemplate=f"<br>"+
                    "%{x}<br>" +
                    "Total Orders: %{y:,}<extra></extra>",
    showlegend=False
), row=1, col=1)

fig.update_xaxes(title="Year_Month",
                 tickmode= "array",
                 tickvals = yearly_month_orders['year_month'].unique(),
                 ticktext = yearly_month_orders['year_month'].unique(),
                 tickangle = -60,
                 showgrid=False, row=1, col=1)
fig.update_yaxes(title="Total Orders", showgrid=False, row=1, col=1)


# Order Distribution by Weekday (Barchart)
fig.add_trace(go.Bar(
    x = weekday_orders['order_day_name'],
    y = weekday_orders['count'],
    text = [f"{c:,}<br>({p}%)" for c, p in zip(weekday_orders['count'], weekday_orders['percentage'])],
    textfont=dict(size=10),
    textposition='outside',
    marker = dict(color=weekday_orders["count"], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
    showlegend = False
), row=2, col=1)

fig.update_xaxes(title="Weekday",
                 categoryorder = 'array',
                 categoryarray = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], row=2, col=1)
fig.update_yaxes(title="Total Orders", showgrid=False, row=2, col=1)
fig.update_traces(
    cliponaxis=False, row=2, col=1)

# Order Distribution by Time of Day (Barchart) 
fig.add_trace(go.Bar(
    x = timely_orders['order_time_day'],
    y = timely_orders['count'],
    text = [f"{c:,}<br>({p}%)" for c, p in zip(timely_orders['count'], timely_orders['percentage'])],
    textfont=dict(size=10),
    textposition='outside',
    marker = dict(color=timely_orders['count'], colorscale=["#89dcb3", "#4ecb8d", "#2e9e67"]),
    showlegend = False
), row=2, col=2)

fig.update_xaxes(title="Time of Day",
                 categoryorder = 'array',
                 categoryarray = ['Morning', 'Afternoon', 'Evening', 'Night', 'Mid-night'], row=2, col=2)
fig.update_yaxes(title="Total Orders", showgrid=False, row=2, col=2)
fig.update_traces(
    cliponaxis=False, row=2, col=2)

#Spacing between Supbplot Title and Chart
for ann in fig['layout']['annotations'][:3]:  # Only adjust subplot titles
    ann['y'] += 0.030

fig.update_layout(height=1000,
                  template="plotly_dark")

fig.show()

<h3>1. Trend over Time: Evolution of Orders:</h3>
E-commerce on Brazil shows strong growth trend from late 2016 to mid-2018, with monthly orders increasing steadily until around mid-2018. This indicates a healthy growth trajectory in customer engagement and online shopping activity this is because customeers are more inclined to shopping online than before.

Noticable Surges and Seasonal Trends:
<li>November 2017 and January 2018 - Peak in the orders are likely due to holiday sales, black friday, end-of-year promotions and start of year sales.
<li>Mid-year (May–August 2018) maintains consistently high volume, suggesting no strong summer dip, unlike in some Western markets.
<li>There's a noticable dip at the end of timeline from September 2018 onwards is because of the noise on data(incomplete data), For futher comparison between the years 2017 and 2018, let's just consider orders between the months January and August in both years.</li>

<h3>Weekly Ordering Patterns: Weekdays vs. Weekends:</h3>
<li>Customer activity is significantly higher during weekdays, with a peak on Monday (16.29%). Customers appear to prefer ordering at the start of the workweek, potentially using workplace time or planning ahead for the week.
<li>Weekend activity drops, indicating that leisure or offline activities may reduce online engagement.</li>

<h3>Time of Day: When Do Customers Place Orders?</h3>
<li>Most orders are placed in the Afternoon (32.39%) and Night (28.49%), making up over 60% of all purchases. Afternoon peaks suggest lunch-hour shopping.
<li>Night-time orders point to post-work convenience</li>
    

<h3>Comparison of orders between year 2017 and 2018:</h3>

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# KPI calculations
orders_2017 = orders[(orders["order_year"] == 2017) & (orders["order_month"] <= 8)]["order_id"].count()
orders_2018 = orders[(orders["order_year"] == 2018) & (orders["order_month"] <= 8)]["order_id"].count()

color_map = {
    2017: "#4ECB8D",
    2018: "#876FD4"
}

# Plotly Subplots
fig = make_subplots(rows=2, cols=2, column_widths=[0.3, 0.7],
                    specs=[[{"type": "domain"}, {"type": "xy", "rowspan" : 2}],  # Row 1: KPI1, Chart top
                           [{"type": "domain"}, None]],  # Row 2: KPI2, Chart bottom
                    vertical_spacing=0)

fig.add_trace(go.Indicator(
    mode="number",
    value=orders_2017,
    number={"valueformat": ",", "font": {"size": 48, "color": "#4ecb8d"}},
    title={"text": "<span style='font-size:20px'>2017 Orders</span><br><span style='font-size:11px'>(Jan–Aug)</span>"}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number+delta",
    value=orders_2018,
    number={"valueformat": ",", "font": {"size": 48, "color" : "#8d4ecb"}},
    delta={"reference": orders_2017, "relative": True, "valueformat": ".0%", "position": "bottom"},
    title={"text": "<span style='font-size:20px'>2018 Orders</span><br><span style='font-size:11px'>(Jan–Aug)</span>"}
), row=2, col=1)

df_filtered = orders.query("order_year > 2016 & order_month <= 8")
bar_data = df_filtered.groupby(["order_year", "order_month_name"], observed=True).size().reset_index(name="order_count")
bar_data["percentage"] = bar_data.groupby("order_year")["order_count"].transform(lambda x: (x / x.sum() * 100).round(1))


for year in sorted(set(bar_data["order_year"])):
    df_year = bar_data[bar_data["order_year"] == year]
    fig.add_trace(go.Bar(
        x=df_year["order_month_name"],
        y=df_year["order_count"],
        name=str(year),
        marker=dict(color=color_map[year]),
        text=[f"{c:,}<br>({p}%)" for c, p in zip(df_year["order_count"], df_year["percentage"])],
        textposition="outside",
        textfont=dict(color=color_map[year], size=10),
        hovertemplate=f"Year: {year}<br>" +
                    "Month: %{x}<br>" +
                    "Total Orders: %{y}<extra></extra>",
        hoverlabel=dict(font=dict(color="white"))
    ), row=1, col=2)

fig.update_xaxes(title_text="Month", showgrid = False)
fig.update_yaxes(title_text="Total Orders", showgrid = False)
fig.update_traces(
    cliponaxis=False, row=1, col=2)

fig.update_layout(
    barmode='group',
    width = 1240,
    height=600,
    #title={"text" :'Orders Comparison Jan–Aug: 2017 vs 2018', "font": {'size': 20, 'color': 'white'}, "xanchor": 'left'},
    xaxis={'categoryorder': 'array', 'categoryarray': ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug"]},
    template="plotly_dark",
    hovermode = "x unified")

fig.show()


<h3>Explosive Year-over-Year Growth (+135%)</h3>

<li>2017 Jan–Aug orders: 22,968

<li>2018 Jan–Aug orders: 53,991</li>

This signals rapid expansion in Brazil's e-commerce and improved site reach marketing efficiency, or operational reliability with strong demand momentum entering 2018. Every month from Jan–Aug 2018 shows 2×–3× higher order volume than the same months in 2017.

<h3>Seasonality Peaks around March-May:</h3>

In both years, March–May are high-volume months (likely tied to holiday periods or promotions). A consistent seasonal spike indicates predictable demand cycles. Brazilian consumers show a strongly early-year purchasing cycle brands can leverage this by increasing ad spend, promotions, and inventory availability in Q1–Q2.

In [ ]:
#Merging all dataset for further analysis:

customer_orders = orders.merge(customers, on= "customer_id")

customer_orders = customer_orders.merge(order_items, on= "order_id")

customer_orders = customer_orders.merge(sellers, on= "seller_id")

customer_orders = customer_orders.merge(products[["product_id", "product_category_name"]], on= "product_id")

customer_orders = customer_orders.merge(products_name, how="left", on="product_category_name")

reviews = reviews.sort_values("review_creation_date").groupby("order_id").tail(1)

customer_orders = customer_orders.merge(reviews[["order_id", "review_score"]], how="left", on="order_id")

geo_location.drop_duplicates(inplace=True, ignore_index=True)

geo_location.drop_duplicates(subset="geolocation_zip_code_prefix", inplace=True)

customer_orders = customer_orders.merge(geo_location, how="left", left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix")

region = pd.read_json('datasets/br.json').rename(columns={"UF-sigla" : "geo_state_id", "UF-nome" : "geo_state", "regiao-nome" : "region_name", "regiao-sigla" : "region_id"})\
    .replace({"Norte" : "North", "Nordeste" : "North East", "Sudeste" : "South East", "Sul" : "South", "Centro-Oeste" : "Central West"})

customer_orders = customer_orders.merge(region[["geo_state", "geo_state_id", "region_name", "region_id"]], how="left", left_on="customer_state", right_on="geo_state_id")

customer_orders.drop(columns=["customer_zip_code_prefix", "product_id", "shipping_limit_date", "seller_zip_code_prefix", "product_category_name", "geolocation_zip_code_prefix"], inplace=True)

customer_orders


<h3>Orders, Seasonability, Growth Analysis:</h3>

Detect seasonality patterns, monthly sales cycles, and whether order growth is structural (long-term) or seasonal (short-term)


In [ ]:
#seasonal index order
monthly_orders = orders.query("order_year > 2016 & order_year_month not in ['2018-09', '2018-10']").groupby(["order_year", "order_year_month"]).agg(order_counts=('order_id','count')).reset_index()
 
avg_orders_per_year = monthly_orders.groupby("order_year").agg(avg_orders = ('order_counts','mean')).reset_index()

seasonal_index_order = monthly_orders.merge(avg_orders_per_year, how='left', on='order_year')

seasonal_index_order['seasonal_index_order'] = (seasonal_index_order['order_counts'] / seasonal_index_order['avg_orders']).round(2)

seasonal_index_order['order_counts'].pct_change() * 100

#seasonal index revenue
monthly_revenue = customer_orders.query("order_year > 2016 & order_year_month not in ['2018-09', '2018-10']").groupby(["order_year", "order_year_month"]).agg(revenue = ('price','sum')).reset_index()

avg_revenue_per_year = monthly_revenue.groupby("order_year").agg(avg_revenue = ('revenue','mean')).reset_index()

seasonal_index_revenue = monthly_revenue.merge(avg_revenue_per_year, how='left', on='order_year')

seasonal_index_revenue['seasonal_index_revenue'] = (seasonal_index_revenue['revenue'] / seasonal_index_revenue['avg_revenue']).round(2)

seasonal_index_revenue



In [ ]:
s_fig= go.Figure()
for i in range(3):
    if i==0:
        s_fig.add_trace(go.Scatter(
        x = seasonal_index_order["order_year_month"],
        y = seasonal_index_order["seasonal_index_order"],
        mode = "lines+markers",
        name = 'seasonal index order',
        line=dict(
            color='#8d4ecb')
))
    if i==1:
        s_fig.add_trace(go.Scatter(
        x = seasonal_index_revenue["order_year_month"],
        y = seasonal_index_revenue["seasonal_index_revenue"],
        mode="lines+markers",
        name='seasonal index revenue',
        line=dict(
            color='#4ecb8d')
))
    else:
        s_fig.add_hline(y=seasonal_index_order['seasonal_index_order'].mean().round(), 
                        line_width=1, 
                        line_dash="dash", 
                        line_color="red", 
                        annotation_text="average monthly volume = Index 1.0", 
                        annotation_position='top')

s_fig.update_xaxes(
    tickmode='array',
    ticks = 'outside',
    tickvals=seasonal_index_order["order_year_month"].unique(),  # full list of months
    ticktext=seasonal_index_order["order_year_month"].unique(),  # same as labels
    tickangle=-60,
    showgrid=False
 )

s_fig.update_yaxes(showgrid=False)

s_fig.update_layout(
    width = 1240,
    height=700,
    title={"text" :'Monthly Orders with Growth Rate and Seasonal Patterns (Jan 2017-Aug 2018)', "font": {'size': 20, 'color': 'white'}, "xanchor": 'left'},
    template="plotly_dark",
    hovermode = 'x unified')
s_fig.show()


<h3>Low seasonal activity in early 2017:</h3>

Seasonal indices for the first quarter range from 0.2 to 0.7, reflecting a typical time of slow sales period.

<h3>Strong seasonal peak from October to December 2017:</h3>
Revenue indices are somewhat outperform order indices, indicating pricing or product mix impacts during this era, driven by important commercial events.

<li>October: Children's and Teachers' Days increase purchases of higher-value categories such as toys and books.
<li>November: Black Friday and Singles' Day cause the greatest sales rise, showing big discounts and broad product purchasing.
<li>December: Eventhough december experiences a sharp drop Christmas and New Year's preparations keeps the orders flowing, particularly for presents and seasonal items, causing revenue to exceed order volume.</li>

<h3>Close alignment between order and revenue seasonality:</h3>
Revenue and order seasonality are well aligned, with a difference of about 0.05-0.1 for most months. This indicates that revenue growth is mostly driven by volume, not pricing or product mix changes.

<h3>More consistent seasonality in 2018:</h3>

Seasonal indices stabilize around 1 (range 0.91 to 1.12), indicating less monthly fluctuation and a maturation towards more stable sales trends with higher baseline order volume and higher revenue index meaning the platform is not just experiencing seasonal spikes; it is growing organically year-over-year.

# Order - Regional Performance Analysis:
Understand how demand is distributed across Brazil to identify high-value markets and regional concentration patterns that may influence logistics, marketing, and revenue strategy.

An overview of customer's order by region, state and city:

In [ ]:
orders_by_region = customer_orders.query("order_year > 2016 & order_year_month != '2018-09'").groupby(["region_name", "order_year_month"])["order_id"].count().reset_index(name="Total Orders")

orders_by_state = customer_orders.groupby("customer_state", observed=True).size().reset_index(name="orders_count")

orders_by_state["percentage"] = orders_by_state["orders_count"].transform(lambda x : (x/x.sum() * 100).round(1))

orders_by_city = customer_orders.groupby("customer_city").size().reset_index(name="orders_count").sort_values(by="orders_count", ascending=False).head(10)

#Plotly Subplots
fig = make_subplots(rows=2, cols=2, column_widths=[1.0, 1.0], row_heights=[0.6, 0.6],
                    specs=[[{"type": "xy"}, {"type": "xy", "rowspan" : 2}],  # Row 1: KPI1 top, KPI bottom
                           [{"type": "xy"}, None]],  # Chart top and bottom
                    subplot_titles=["Evolution of E-Commerce Orders on Brazil Regions Over Years", "Total Orders by State", "Top 10 Brazil Cities With Highest Orders"],
                    vertical_spacing=0.15)

# Line Chart (Orders by region)
for region in orders_by_region["region_name"].unique():
    region_df = orders_by_region[orders_by_region["region_name"] == region]
    fig.add_trace(go.Scatter(
        x = region_df["order_year_month"],
        y = region_df["Total Orders"],
        mode="lines+markers",
        name=region,
        cliponaxis=False
), row=1, col=1)

fig.update_xaxes(
    tickmode='array',
    ticks = 'outside',
    tickvals=orders_by_region["order_year_month"].unique(),  # full list of months
    ticktext=orders_by_region["order_year_month"].unique(),  # same as labels
    tickangle=-60,
    showgrid=False,
    row=1, col=1 
)

fig.update_yaxes(title="Orders", showgrid=False, row=1, col=1)

# Bar Chart (Orders by City)
sorted_city = orders_by_city.sort_values("orders_count")    
fig.add_trace(go.Bar(
    x=sorted_city["orders_count"],
    y=sorted_city["customer_city"],
    orientation='h',
    marker = dict(color=sorted_city['orders_count'], colorscale=["#b389dc", "#8d4ecb","#672e9e"]),
    text = [f'{c}' for c in sorted_city["orders_count"]],
    textposition='outside',
    showlegend=False,
    cliponaxis=False
),row=2, col=1)   

fig.update_xaxes(title="Orders", showgrid=False, row=2, col=1)

# Bar Chart (Orders by State)
sorted_state = orders_by_state.sort_values("orders_count")
fig.add_trace(go.Bar(
    x=sorted_state["orders_count"],
    y=sorted_state["customer_state"],
    orientation='h',
    marker = dict(color=sorted_state['orders_count'], colorscale=["#89dcb3", "#4ecb8d", "#2e9e67"]),
    text = [f'{c}({p})' for c,p in zip(sorted_state["orders_count"], sorted_state["percentage"])],
    textposition="outside",
    showlegend=False,
    cliponaxis=False
), row=1, col=2)

fig.update_xaxes(title="Orders", showgrid=False, row=1, col=2)

#Spacing between Supbplot Title and Chart
for ann in fig['layout']['annotations'][:3]:  # Only adjust subplot titles
    ann['y'] += 0.015

# Step 1: Safely retrieve existing subplot titles (auto annotations)
existing_annotations = fig.layout.annotations if 'annotations' in fig.layout else []

# Step 2: custom annotation
custom_annotation = dict(
    x=orders_by_region.loc[orders_by_region["Total Orders"].idxmax()]["order_year_month"],
    y=orders_by_region["Total Orders"].max(),
    xref='x',
    yref='y',
    text='Highest Orders',
    showarrow=True,
    ax=0,
    ay=-40,
    font=dict(color='white')  # optional styling
)

# Step 3: Apply layout update with title and annotation
fig.update_layout(
    width=1240,
    height=900,
    title=dict(
        #text='Overview of Orders by Region, State and City',
        font=dict(size=21, color='white'),
        xanchor='left'
    ),
    legend=dict(
        x=0.01,
        y=0.99,
        yanchor='top',
        xanchor='left',
        bgcolor='rgba(0,0,0,0)'
    ),
    template="plotly_dark",
    annotations=fig.layout.annotations + (custom_annotation,)  #preserve subplot titles
)
    
fig.show()

<h3>Southeast is the Economic Engine of Brazilian E-commerce</h3>

The South eastern region of Brail exhibits the most significant growth in orders, recording an all-time high of 5,933 orders in November 2017.Additionally, a noticable trend of order peaks is observed from October to December across all regions, accompanied by a consistent flow of orders throughout 2018.

Demand is highly centralized: Undoubtedly, two prominent cities in brazil  are São Paulo the largest city and economic center with the greatest orders of 17,808 and Rio de Janeiro renowned for it's tourism has recorded the second highest orders of 7,837. These two cities alone account for more than half of all national orders. This concentration mirrors Brazil’s population density and economic activity, but also reveals a heavy dependency on just a few states to drive revenue.

<h3>All Regions Show Growth, but None Come Close to Southeast’s Scale</h3>

Although North, Northeast, South, and Central-West exhibit consistent year-over-year order growth, their volumes remain significantly smaller. This suggests untapped market potential where targeted marketing or faster delivery could unlock additional demand.

<h3>Urban Hubs Dominate Purchase Behavior</h3>

Cities like São Paulo, Rio de Janeiro, Belo Horizonte, and Brasília lead by a large margin, confirming that e-commerce adoption is strongest in dense metropolitan areas. Mid-tier cities (Campinas, Porto Alegre, Salvador) also show meaningful demand but with slower momentum.


<h3>Distribution of Customers in Brazil States:</h3>

In [ ]:
data = requests.get("https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson").json()
states = gpd.GeoDataFrame.from_features(data, crs="EPSG:4326")

states = states[["geometry", "name", "sigla"]].rename(columns={'sigla': 'customer_state'})
brazil_states = states.merge(orders_by_state, on="customer_state")

mymap = folium.Map(location=[-15.788497, -47.879873], zoom_start=4,tiles=None)
folium.TileLayer('CartoDB positron',name="Light Map",control=False).add_to(mymap)

folium.Choropleth(
    geo_data=brazil_states,
    data=brazil_states,
    columns=['customer_state','orders_count'],
    fill_color="Spectral",
    bins=[0, 10000, 20000, 30000, 40000, orders_by_state["orders_count"].max()],
    line_opacity=0.2,
    legend_name='Customer Orders by State',
    smooth_factor=0,
    key_on="feature.properties.customer_state",
    highlight=True).add_to(mymap)

style_function = lambda x: {'fillColor': '#ffffff', 
                            'color':'#000000', 
                            'fillOpacity': 0.1, 
                            'weight': 0.1}
highlight_function = lambda x: {#'fillColor': '#000000', 
                                #'color':'#000000', 
                                'fillOpacity': 0.50, 
                                'weight': 0.1}

NIL = folium.features.GeoJson(
    brazil_states,
    style_function=style_function, 
    control=False,
    highlight_function=highlight_function, 
    tooltip=folium.features.GeoJsonTooltip(
        fields=['name','orders_count', 'percentage'],
        aliases=['Customer State: ','Total Orders: ','Total Orders in %: '],
        style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;") 
    )
)
#folium.GeoJson(brazil_geo).add_to(mymap)
mymap.add_child(NIL)
mymap.keep_in_front(NIL)
folium.LayerControl().add_to(mymap)

mymap

<h4>In-depth View of Customer Orders from Brazil showing every region for 2017 and 2018:</h4>

In [ ]:
fig = branca.element.Figure()

map_2017 = map_2017 = fol = customer_orders.query("order_year == 2017")[["customer_city", "order_year", "geolocation_lat", "geolocation_lng"]]
map_2017.dropna(inplace=True)
map_2018 = fol = customer_orders.query("order_year == 2018 & order_year_month != '2018-09'")[["customer_city", "order_year", "geolocation_lat", "geolocation_lng"]]
map_2018.dropna(inplace=True)

subplot1 = fig.add_subplot(1, 2, 1)
subplot2 = fig.add_subplot(1, 2, 2)

m1 = folium.Map(location=[-15.788497, -47.879873], zoom_start=4)

marker_cluster1 = MarkerCluster().add_to(m1)

m2 = folium.Map(location=[-15.788497, -47.879873], zoom_start=4)

marker_cluster2 = MarkerCluster().add_to(m2)

for i, row in map_2017.iterrows():
    folium.CircleMarker(
        location=[row["geolocation_lat"], row["geolocation_lng"]],
        radius=5,  # size of the point
        popup=f"{row['customer_city']}",
        color='orange',
        fill=True,
        fill_color='orange',
        fill_opacity=0.7
    ).add_to(marker_cluster1)

for i, row in map_2018.iterrows():
    folium.CircleMarker(
        location=[row["geolocation_lat"], row["geolocation_lng"]],
        radius=5,  # size of the point
        popup=f"{row['customer_city']}",
        color='orange',
        fill=True,
        fill_color='orange',
        fill_opacity=0.7
    ).add_to(marker_cluster2)

subplot1.add_child(m1)
subplot2.add_child(m2)

title_html = '''
             <h3 align="center" style="font-size:20px"><b>Customer Orders in 2017 at Brazil</b></h3>
             '''
m1.get_root().html.add_child(folium.Element(title_html))

title_html = '''
             <h3 align="center" style="font-size:20px"><b>Customer Orders in 2018 at Brazil</b></h3>
             '''
m2.get_root().html.add_child(folium.Element(title_html))

fig

# Delivery Performance Analysis:

<h3>Objective:</h3>

To understand how effectively orders are fulfilled by analyzing:

<li>How long deliveries actually take

<li>How accurate estimated delivery promises are

<li>How delivery performance varies across Brazilian regions & states</li>

This helps identify logistics bottlenecks, regional delays, and opportunities for improving delivery promise accuracy.

In [ ]:
fig = make_subplots(rows= 1, cols=2,
                    specs = [[{"type":"xy"}, {"type": "xy"}]],
                    subplot_titles=["Distribution of Actual Delivery Days", "Difference between Estimated and Actual Delivery"])

fig.add_trace(go.Histogram(x = orders["delivery_days"], nbinsx=40, 
                           marker=dict(color='#4ecb8d', line=dict(color='black', width=1)), showlegend=False),row=1, col=1)
fig.update_xaxes(title = "Days to deliver", row=1, col=1)
fig.update_yaxes(title = "Number of Orders", showgrid=False, row=1, col=1)

fig.add_trace(go.Histogram(x = orders["delay_vs_estimate"], nbinsx=40, 
                           marker=dict(color= '#8d4ecb', line=dict(color='black', width=1)), showlegend=False), row=1, col=2)
fig.add_vline(x = 0, line_width = 1.5, line_dash ="dash", line_color = "green", row=1, col=2)
fig.update_xaxes(title = "Days Late or Early", row=1, col=2)
fig.update_yaxes(title = "Number of Orders", showgrid=False, row=1, col=2)

fig.update_layout(height = 600, 
                  template="plotly_dark")
fig.show()


<h3>Distribution of Delivery Days:</h3>

Most orders are delivered within 5–20 days, forming a strong right-skewed distribution.

A small proportion of orders take 40+ days, and a few extreme outliers exceed 100 days, indicating occasional severe delays.

The core takeaway: delivery performance is generally stable, but long-tail delays exist and inflate customer dissatisfaction risk.

The recommended action is to investigate outlier deliveries to identify whether delays are due to remote locations, seller-side fulfillment lag or carrier inefficiencies.

<h3>Estimated vs Actual Delivery — Accuracy of Delivery Promises:</h3>

The histogram of differences is tightly clustered around zero but with a noticeable negative tail: most orders arrive near or slightly before estimate, but a subset experience sizable lateness (again, outliers drive the problem).

Implication: estimate quality is generally fine, but exceptions (large negative deviations) suggest particular routes/regions need operational fixes.


# Regional Delivery Performance 

Let's dig deeper into how delivery times vary across:

<li>Customer Locations (Region and States) – Are some regions experiencing delays more often?
<li>Top Sellers – Who delivers faster? Who's consistently late?</li>
    



In [ ]:
state_delivery = (
    customer_orders
    .groupby(['region_name','geo_state'])['delivery_days']
    .mean()
    .sort_values()
    .round()
    .astype(int)
    .reset_index(name='avg_delivery_days')
)

color_map = {
    'North': "#a6d854",
    'North East': "#fc8d62",
    'Central West': "#8da0cb",
    'South East': "#e78ac3",
    'South': "#66c2a5"
}

fig = go.Figure()

for regions in set(state_delivery["region_name"]):
    df_state = state_delivery[state_delivery["region_name"] == regions]
    fig.add_trace(go.Bar(
        x=df_state["geo_state"],
        y=df_state["avg_delivery_days"],
        name=str(regions),
        marker=dict(color=color_map[regions]),
        text= [f"{c}" for c in df_state["avg_delivery_days"]],
        textposition="outside",
        textfont=dict(color=color_map[regions], size=10),
        hovertemplate=f"Region: {regions}<br>" +
                    "State: %{x}<br>" +
                    "Avg Delivery Days: %{y}<extra></extra>")
    )

fig.update_xaxes(title_text="Customer State", showgrid = False, tickangle = -45)
fig.update_yaxes(title_text="Avg Delivery Days", showgrid = False)

fig.update_layout(
    barmode='group',
    width = 1240,
    height=600,
    title={"text" :'Average Delivery Days by Cities in Brazil', "font": {'size': 20, 'color': 'white'}, "xanchor": 'left'},
    template = "plotly_dark")

fig.show()

<h3>Average Delivery Days by State — Insights:</h3>

Southeast and South regions perform best (São Paulo, Paraná, Santa Catarina) with 8–15 day averages.

North and Northeast regions experience the longest delays (Amazonas, Amapá, Roraima at 26–28 days).

This gap shows strong geographic inequality in logistics performance—driven by infrastructure challenges and distance from distribution hubs.

the recommended action is consider regional fulfillment strategies, such as partnering with local carriers in North/Northeast, pre-positioning inventory in regional hubs and offering dynamic shipping fees to offset long-distance constraints.

In [ ]:
with open('datasets/brazil_states.json', 'r') as file:
    data = json.load(file)

avg_delivery= customer_orders.groupby(["geo_state"])["delivery_days"].mean().round().astype(int).reset_index(name="avg_delivery_days")

geo_df = gpd.GeoDataFrame.from_features(data["features"]).merge(avg_delivery, left_on="name", right_on="geo_state").set_index("geo_state")

fig = px.choropleth(geo_df,
                   geojson=geo_df.geometry,
                   locations=geo_df.index,
                   color="avg_delivery_days",
                   color_continuous_scale='Viridis')
                   #projection="mercator")
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(title = "Geographic Delivery Performance: Southeast Delivers Faster", template = "plotly_dark")
fig.show()

<h3>Average Early vs Delayed Deliveries by Region and States in Brazil</h3>

In [ ]:
region_early = customer_orders.query("delay_vs_estimate >= 0").groupby("region_name")["delay_vs_estimate"].mean().round().astype(int).sort_values(ascending=False).reset_index()
region_delay = customer_orders.query("delay_vs_estimate < 0").groupby("region_name")["delay_vs_estimate"].mean().round().astype(int).reset_index()

fig = make_subplots(rows=2, cols=1, column_widths=[1.0], row_heights=[0.6, 0.6],
                    subplot_titles=["Average Early vs Delayed Deliveries by Regions in Brazil", "Average Early vs Delayed Deliveries by States in Brazil"],
                    vertical_spacing=0.15)

#Plot-1

fig.add_trace(go.Bar(x=region_early['region_name'], y=region_early['delay_vs_estimate'],
                marker_color='#4ECB8D',
                name='Early'
                ), row=1, col=1)
fig.add_trace(go.Bar(x=region_delay['region_name'], y=region_delay['delay_vs_estimate'],
                marker_color='#876FD4',
                name='Delay'),row=1, col=1)

#Plot-2

states_delayed = customer_orders.query("delay_vs_estimate < 0").groupby("geo_state")["delay_vs_estimate"].mean().round().astype(int).reset_index()

states_faster = customer_orders.query("delay_vs_estimate >= 0").groupby("geo_state")["delay_vs_estimate"].mean().round().astype(int).sort_values(ascending=False).reset_index()


fig.add_trace(go.Bar(x=states_faster['geo_state'], y=states_faster['delay_vs_estimate'],
                marker_color='#4ECB8D',
                name='Early',
                showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=states_delayed['geo_state'], y=states_delayed['delay_vs_estimate'],
                marker_color='#876FD4',
                name='Delay',
                showlegend=False), row=2, col=1)

fig.update_xaxes(title_text="State", tickangle = -55, showgrid=False, row=2, col=1)
fig.update_yaxes(title_text="Delay vs Estimate", row=2, showgrid=False, col=1)

fig.update_xaxes(title_text="Region", row=1, col=1)
fig.update_yaxes(title_text="Delay vs Estimate", row=1, col=1)

fig.update_layout(
    width=1240,
    height=900,
    xaxis = dict(showgrid=False),
    yaxis = dict(showgrid=False),
    template = "plotly_dark",
    hovermode="x unified"
)
fig.show()


South-East and Central-West show meaningful early-delivery performance on average, while North and North-East show larger average delays compared with estimates and overall.

Implication: Logistics capacity and infrastructure differences (distance, infrastructure, carrier coverage) are likely driving systematic regional gaps.

Moreover all the states have a good average delivery perfomance(10-20 days) above the estimated time with a few exceptions, like Amapa(-73), Roraima(-37) and Acre(-20) which may be due to their mixed cultures of modern and remote areas and underdeveloped infrastructure, and all those states are mostly situated in northern part of brazil which has expreienced the highest delivery delay compared to other regions of which the above regional chart proves 

<h4>Let's examine the top five Brazilian states in terms of delivery times.</h4>

In [ ]:
mean_fast_delivery = customer_orders.query("delay_vs_estimate >= 0")["delay_vs_estimate"].mean()

mean_delay_delivery = customer_orders.query("delay_vs_estimate < 0")["delay_vs_estimate"].mean()

mean_delivery_days = customer_orders["delivery_days"].mean()

mean_delivery_days


In [ ]:
top5_delay = customer_orders.query("delay_vs_estimate < 0").groupby(["customer_state", "geo_state"])["delay_vs_estimate"].mean().round().astype(int).nsmallest(5).sort_values(ascending=False).reset_index(name="avg_delay")

top5_fast = customer_orders.query("delay_vs_estimate > 0").groupby(["customer_state", "geo_state"])["delay_vs_estimate"].mean().round().astype(int).nlargest(5).sort_values().reset_index(name="avg_fast")

top5_avg_delivery = customer_orders.groupby(["customer_state", "geo_state"])["delivery_days"].mean().round().astype(int).nsmallest(5).sort_values().reset_index(name="avg_delivery")

fig = make_subplots(rows=2, cols=3,
                    specs = [[{"type":"domain"}, {"type":"domain"}, {"type":"domain"}],
                             [{"type":"xy"}, {"type":"xy"}, {"type":"xy"}]],
                    row_heights=[0.3, 0.8],
                    subplot_titles=[None, None, None, "<span style='font-size:13px'>Top 5 States with<br>quickest average time to delivery</span>", 
                                    "<span style='font-size:13px'>Top 5 states with faster delivery<br>compared to the estimated time</span>", 
                                    "<span style='font-size:13px'>Top 5 States with larger delays<br>compared to estimated time</span>"],
                    vertical_spacing=0.4, horizontal_spacing=0.1)

#Indicator-1
fig.add_trace(go.Indicator(mode="number",
                           value = int(mean_delivery_days.round()),
                           number={"suffix":"<span style='font-size:20px'> working days</span>", "font":{'size': 40, 'color': '#4ECB8D'}}),row=1,col=1)

fig.add_annotation(
    text="is the average delivery time",
    xref="paper", yref="paper",
    x=0.05, y=0.85,  # Adjust coordinates as needed
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="center")

#Indicator-2
fig.add_trace(go.Indicator(mode="number",
                           value = int(mean_fast_delivery.round()),
                           number={"suffix":"<span style='font-size:20px'> working days</span>", "font":{'size': 40, 'color': '#4ECB8D'}}),row=1,col=2)

fig.add_annotation(
    text="is the average difference between<br>estimated and delivery time(fast delivery)",
    xref="paper", yref="paper",
    x=0.51, y=0.85,  # Adjust coordinates as needed
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="center")

#Indicator-3
fig.add_trace(go.Indicator(mode="number",
                           value = int(mean_delay_delivery.round()),
                           number={"suffix":"<span style='font-size:20px'> working days</span>", "font":{'size': 40, 'color': '#8d4ecb'}}),row=1,col=3)

fig.add_annotation(
    text="is the average delay time to deliver<br>after the estimated time(late delivery)",
    xref="paper", yref="paper",
    x=0.98, y=0.85,  # Adjust coordinates as needed
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="center")

#Bar-1
fig.add_trace(go.Bar(x = top5_avg_delivery["avg_delivery"],
                     y = top5_avg_delivery["customer_state"],
                     orientation='h',
                     hovertemplate="%{y}<br>"+ "Avg delivery days: %{x}<extra></extra>",
                     marker = dict(color=top5_avg_delivery["avg_delivery"], colorscale=["#89dcb3", "#4ecb8d", "#2e9e67"]),
                     showlegend=False),row=2, col=1)

fig.update_xaxes(title = "avg delivery days", showgrid=False, row=2, col=1)

#Bar-2
fig.add_trace(go.Bar(x = top5_fast["avg_fast"],
                     y = top5_fast["customer_state"],
                     orientation='h',
                     hovertemplate="%{y}<br>"+ "Avg fast delivery days: %{x}<extra></extra>",
                     marker = dict(color=top5_fast["avg_fast"], colorscale=["#89dcb3", "#4ecb8d", "#2e9e67"]),
                     showlegend=False),row=2, col=2)

fig.update_xaxes(title = "avg fast delivery days", showgrid=False, row=2, col=2)

#Bar-3
fig.add_trace(go.Bar(x = top5_delay["avg_delay"],
                     y = top5_delay["customer_state"],
                     orientation='h',
                     hovertemplate="%{y}<br>"+ "Avg late delivery days: %{x}<extra></extra>",
                     marker = dict(color=top5_delay["avg_delay"], colorscale=["#672e9e", "#8d4ecb", "#b389dc",]),
                     showlegend=False),row=2, col=3)

fig.update_xaxes(title = "avg late delivery days", showgrid=False, row=2, col=3)

fig.update_layout(title = {"text" :'A Comparative Analysis: E-commerce Delivery Performance in Brazilian States', "font": {'size': 20, 'color': 'white'}},
                  template = "plotly_dark",
                  height = 450)

fig.show()



<h3>State & city-level takeaways:</h3>

Top 5 fastest average delivery times: ES, DF, MG, PR, SP — these states consistently deliver quicker on average (good benchmark for operational practices).

Top 5 states that beat estimates most often (arrive early): RR, AC, AP, AM, RO — these states show positive early vs estimate performance (maybe conservative estimates or strong local logistics).

Top 5 states with largest average lateness (biggest negative deviations): AP, RR, AC, AM, SE (as shown in the comparative chart).
Note: the charts show both early and late extremes — some states show both strong early performance for some orders and large late outliers for others → indicates high variability and inconsistent logistic reliability.

Cities: São Paulo dominates order volume. High-volume hubs (São Paulo / Rio / Belo Horizonte) are operationally critical: improving performance here yields big customer-experience improvements.

# Payment Type Analysis:
<h3>Objective:</h3>

Understand how customers pay for their orders, how payment behavior changed from 2017 to 2018, which methods drive higher spending, and how installment usage reflects consumer affordability and buying patterns in Brazilian e-commerce.

In [ ]:
order_payments = customer_orders.merge(payments, on='order_id')
total_payments = order_payments.groupby('payment_type').size().reset_index(name='payments')
total_payments["percentage"] = total_payments["payments"].transform(lambda x : (x/x.sum() * 100).round(1))

usage_share_percent = order_payments.query('order_year > 2016').groupby(["order_year", "payment_type"], observed=True).size().reset_index(name="payment_count")
usage_share_percent["percentage"] = usage_share_percent.groupby("order_year")["payment_count"].transform(lambda x: (x / x.sum() * 100).round(1))
usage_share_percent = usage_share_percent.sort_values(by='percentage', ascending=False)

avg_spending = order_payments.groupby('payment_type')['payment_value'].mean().astype(int).reset_index().sort_values(by='payment_value',ascending=True)

payment_installments = order_payments.groupby('payment_installments').size().reset_index(name='count')
payment_installments['percentage'] = payment_installments['payment_installments'].transform(lambda x: (x/x.sum() * 100).round(1))

fig = make_subplots(rows=2, cols=2, column_widths=[1.0, 1.0], row_heights=[0.4, 0.6],
                    specs=[[{"type":"domain"}, {"type": "xy"}], 
                           [{"type": "xy"}, {"type":"xy"}]],
                    subplot_titles=["Transactions by Payment Type", "Payment Method Usage and YOY% Comparison", "Average Spending by Payment Type", "Payment Installments Distribution"],
                    vertical_spacing=0.15)

color_map = {
    2017: "#4ECB8D",
    2018: "#876FD4"
}

colors = ['#876FD4', '#4ECB8D', '#F0FFF1', '#C2F8CB']
marker_col = ['#C2F8CB', '#F0FFF1', '#4ECB8D', '#876FD4']


#Total payments by payment type

fig.add_trace(go.Pie(labels=total_payments['payment_type'], 
                     values=total_payments['payments'], 
                     name="Payment Type", 
                     hole=.6,
                     marker_colors=colors,
                     text=[f"{c}<br>({p:,})" for c, p in zip(total_payments["payment_type"], total_payments["payments"])],
                     textposition='outside',
                     showlegend=False),
              row=1, col=1)

fig.add_annotation(
    text=f"{total_payments['payments'].sum()}<br>payments<br>registered",
    x=0.225, y=0.87,  # MOVE annotation lower + left so it doesn't kill the title
    xref='paper',
    yref='paper',
    xanchor="center",
    showarrow=False,
    font=dict(size=14, color="white")
)
fig.update_traces(hoverinfo='label+percent+name', row=1, col=1)

# usage_share_percent_by_year

for year in sorted(set(usage_share_percent["order_year"])):
    usage_share_per_year = usage_share_percent[usage_share_percent["order_year"] == year]
    fig.add_trace(go.Bar(
        x=usage_share_per_year["payment_type"],
        y=usage_share_per_year["percentage"],
        name=str(year),
        marker=dict(color=color_map[year]),
        text=[f"{c:,}<br>({p}%)" for c, p in zip(usage_share_per_year["payment_count"], usage_share_per_year["percentage"])],
        textposition="outside",
        textfont=dict(color=color_map[year], size=10),
        hovertemplate=f"Year: {year}<br>" +
                    "Payment Type: %{x}<br>" +
                    "Payment %: %{y}<extra></extra>"), row=1, col=2)
    
fig.update_xaxes(title_text = "Payment Type", showgrid=False, row=1,col=2)
fig.update_yaxes(title_text = "Usage Share %", showgrid=False, row=1,col=2)
    
# Average Spend by Payment Method

# Lollipop = bar line + circle marker
fig.add_trace(go.Scatter(
    x=avg_spending['payment_type'],
    y=avg_spending['payment_value'],
    mode="lines",
    line=dict(color="#888", width=2),
    showlegend=False
    ), row=2, col=1)

fig.add_trace(go.Scatter(
    x=avg_spending['payment_type'],
    y=avg_spending['payment_value'],
    mode="markers+text",
    marker=dict(size=16, color=marker_col, line=dict(width=1, color="white")),
    text=[f"{p} BRL" for p in avg_spending["payment_value"]],
    textposition="middle right",
    hovertemplate="%{x}<br>%{y}<extra></extra>",
    showlegend=False
    ), row=2, col=1)

fig.update_xaxes(title="Payment Type", showgrid=False, row=2, col=1)
fig.update_yaxes(title="Average Spending (BRL)", showgrid=False, row=2, col=1)

#Distribution of Payment Installments

desired_order = payment_installments.sort_values('count', ascending=False)['payment_installments'].to_list()

fig.add_trace(go.Bar(
    x=payment_installments['count'],
    y=payment_installments['payment_installments'],
    orientation='h',
    text=[f"{c:,} ({p}%)" for c, p in zip(payment_installments["count"], payment_installments["percentage"])],
    textposition="outside",
    textfont=dict(size=10),
    hovertemplate="Payments: %{x}<extra></extra>",
    marker = dict(color=payment_installments["count"], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
    showlegend=False),row=2, col=2)

fig.update_xaxes(title="Count",  showgrid=False, row=2, col=2)
fig.update_yaxes(title = "Payment Installments", 
                 autorange="reversed",
                 tickvals = desired_order,
                 ticktext = desired_order,
                 categoryorder ='array', 
                 categoryarray = desired_order,
                 showgrid=False, row=2, col=2)

fig.update_traces(cliponaxis=False, row=1,col=2)  # Allow text to render outside plot area
fig.update_traces(cliponaxis=False, row=2,col=1)
fig.update_traces(cliponaxis=False, row=2,col=2)


#Spacing between Supbplot Title and Chart
for ann in fig['layout']['annotations'][:4]:  # Only adjust subplot titles
    ann['y'] += 0.030


fig.update_layout(height=1000,
                  template="plotly_dark"),

fig.show()


<h3>Credit cards are most favoured</h3>

<li>Credit cards continued to be the most popular form of payment, rising from 72.6% in 2017 to 74.8% in 2018.

<li>This modest increase supports its position as the technique with the greatest average spending per order (179 BRL) and validates its dominant position.

<li>Credit cards are widely used for online transactions because they are fast and secure, allowing customers to complete transactions quickly and with confidence. Additionally, credit cards offer benefits such as rewards programs, cashback, and purchase protection, which can incentivize customers to use them.</li>

<h3>Boleto</h3>
<li>Usage of Boleto decreased slightly from 20.4% in 2017 to 18.6% in 2018.

<li>Customers with substantial purchasing power continue to utilize it, as seen by its solid average order value of 177 BRL despite this decline.

<li>Boletos are another popular payment type in Brazil, especially for customers who do not have access to credit cards or do not feel comfortable using them for online transactions. Boletos are essentially bank vouchers that customers can pay at any bank, post office, or convenience store. They are easy to use and can be used by customers who do not have a bank account or a credit card.</li>

<h3>Debit Cards</h3>
Among all payment options, debit cards experienced the fastest rise, rising from 0.9% to 1.9% between 2017 and 2018. With an average order value of 149 BRL, its increasing uptake might be a result of a growing number of younger or recently banked consumers.

<h3>Vouchers</h3>
Vouchers are the least common option, with usage falling from 6.1% in 2017 to 4.6% in 2018. Additionally, it had the lowest average order value (64 BRL), indicating that it is primarily utilized for promotional or low-value transactions.

<h3>Installments are a fundamental part of Brazil’s online purchasing culture</h3>

A massive share of credit-card transactions use installments — with 1, 2, and 3 installments dominating, and long-tail options extending up to 24 installments. This confirms brazilians prefer spreading payments even for medium-value orders, instalment flexibility drives higher conversion and larger basket sizes, merchants without installment options risk lower purchase completion.

# Product Performance:
<h4>Objective:</h4>

To understand which product categories drive the most revenue, how quantity sold aligns with revenue contribution, and which categories expanded or declined between 2017–2018. This helps identify high-value categories, rising opportunities, and declining segments for strategic decision-making.

In [ ]:
products_sold = customer_orders.groupby("product_category_name_english").agg(top_pro_sales=('price','sum'), quantity_sold=('order_item_id','count')).astype(int).nlargest(10, columns='top_pro_sales').sort_values(by='quantity_sold',ascending=False).reset_index()

fig = make_subplots(specs=[[{"secondary_y" : True}]])

fig.add_trace(go.Bar(x = products_sold['product_category_name_english'], 
                     y = products_sold['quantity_sold'],
                     name = "Quantity_sold",
                     text=products_sold['quantity_sold'],
                     textposition='inside',
                     marker=dict(color=products_sold['quantity_sold'], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
                     hovertemplate = "Quantity_sold: %{y:,.0f}<extra></extra>",   
                     offsetgroup = 1),secondary_y=False)

fig.add_trace(go.Scatter(x = products_sold['product_category_name_english'],
                     y = products_sold['top_pro_sales'],
                     name = "Revenue",
                     marker_color = "#4ECB8D",
                     mode='lines+markers+text',
                     text=products_sold['top_pro_sales'],
                     texttemplate="%{y:.2s}",
                     textposition='bottom center',
                     hovertemplate = "Sales: %{y:$,.0f}<extra></extra>",
                     offsetgroup = 2),secondary_y=True)

fig.update_xaxes(title_text = "Product_Categories", showgrid=False)
fig.update_yaxes(title_text = "Quantity Sold", secondary_y=False, showgrid=False)
fig.update_yaxes(title_text = "Revenue <b>(in R$)</b>", secondary_y=True, showgrid=False)

fig.update_layout(title = dict(text = "<span style='font-size:20px'>Top 10 Product Categories (2016-2018): Revenue vs Quantity Sold</span>"),
                  height = 700,
                  template = "plotly_dark",
                  hovermode = 'x unified')
fig.show()


<h3>Pricing Pattern</h3>
The pattern with some categories like (bed_bath_table and furniture_decor), which have low margins but strong volume. This is because the average price of these categories is relatively low (93 BRL for bed_bath_table and 87 BRL for furniture_decor). bed_bath_table, health_beauty, and sports_leisure dominate in both sales volume and revenue, making them the platform’s most stable and reliable growth engines. health_beauty stands out with high demand and premium pricing, suggesting strong brand or product loyalty.

While Other categories like (watches_gifts, health_beauty) generate high revenue despite lower order volume indicating high average selling prices (ASP) (201 BRL for watches_gifts and 130 BRL for health_beauty) and strong margins.

Other small categories contributes to overall income without posing significant risk, demonstrating that the corporation efficiently monitizes multiple niches rather than depending on a few standout goods.

<h3>Business Implication</h3>

Focus marketing and cross-selling efforts around top categories (health & beauty, home goods) while exploring price optimization and bundling opportunities for premium categories like watches & gifts.

<h3>Balanced Diversification</h3>
No single category surpasses the 11% of total revenue and substantially contributes, thus if one segment drops, the entire business impact will be low.

The corporation avoids reliance on a single product category that may risk income, thus diversification lowers business risks and promotes stable growth.



<h3>Top 10 and Bottom 10 Categories by Revenue Share Growth:</h3>

In [ ]:
# Calculate total revenue per category per year
category_revenue = customer_orders.groupby(['order_year', 'product_category_name_english'])['price'].sum().reset_index()

# Calculate total revenue per year
total_revenue_per_year = customer_orders.groupby('order_year')['price'].sum().reset_index(name='total_revenue')

# Merge to calculate revenue share
category_revenue = pd.merge(category_revenue, total_revenue_per_year, on='order_year')
category_revenue['revenue_share'] = (category_revenue['price'] / category_revenue['total_revenue']) * 100

revenue_2017 = category_revenue[category_revenue['order_year'] == 2017].rename(columns={'revenue_share':'revenue_share_2017'})
revenue_2018 = category_revenue[category_revenue['order_year'] == 2018].rename(columns={'revenue_share':'revenue_share_2018'})

revenue_per_category = revenue_2017[['product_category_name_english', 'revenue_share_2017']].merge(revenue_2018[['product_category_name_english', 'revenue_share_2018']], how='inner', on='product_category_name_english')

revenue_per_category['growth_percentage'] = (((revenue_per_category['revenue_share_2018'] - revenue_per_category['revenue_share_2017']) / revenue_per_category['revenue_share_2017']) * 100).round(2)
top_10_revenue_growth = revenue_per_category.nlargest(10, columns='growth_percentage')
bottom_10_revenue_growth = revenue_per_category.nsmallest(10, columns='growth_percentage')

revenue_growth_share = pd.concat([top_10_revenue_growth, bottom_10_revenue_growth])


custom_colors = [[0,"#b389dc"], [0.5, "#89dcb3"], [1, "#4ecb8d"]]
fig = go.Figure(data=[go.Bar(x=revenue_growth_share['product_category_name_english'], y=revenue_growth_share['growth_percentage'],
                             text=revenue_growth_share['growth_percentage'],
                             textposition = 'outside',
                             marker=dict(color=revenue_growth_share['growth_percentage'], colorscale=custom_colors),
                             hovertemplate="%{x}<br>%{y}<extra></extra>")])


fig.update_yaxes(showgrid=False)

# Customize the layout
fig.update_layout(
    title="Top and Bottom 10 Product Categories by Revenue Share Growth (2017-2018)",
    height=700,
    xaxis_title= "Product Category",
    yaxis_title= "Revenue Share Growth (%)",
    template="plotly_dark")

fig.show()


<h3>Explosive Growth in Few Categories:</h3>

small_appliances_home_oven_and_coffee shows a massive 4600%+ growth, making it by far the fastest-growing category likely driven by rising household spending and lifestyle upgrades.

Other high-growth categories such as diapers_and_hygiene and arts_and_craftmanship indicate strong demand shifts toward home & personal care niches.

Construction-related segments (construction_tools, lighting, signaling, safety) also show consistent positive growth, suggesting strong demand from home-improvement & DIY consumers.

The bottom 10 categories like fashion and hardware accessories represent obsolete product segments that are declining rapidly and consistently

<h3>Business Implication</h3>

Strong potential for increasing inventory depth, launching targeted campaigns, or forming vendor partnerships in these rapidly growing segments.

<h3>Comparison of Product Categories Contributing to 80% of Revenue (2017 vs 2018)</h3>

<h4>Objective:</h4>

To compare how the composition of high-value product categories evolved between 2017 and 2018, focusing on:

<li>Which categories continued to dominate revenue,

<li>Which gained revenue share (growing consumer demand), and

<li>Which lost share (declining relevance or increased competition).</li>

This is essential for product assortment planning, inventory strategy, and category investment decisions.

In [ ]:
cat_revenue_2017 = revenue_2017.sort_values('price', ascending=False).reset_index(drop=True)
cat_revenue_2017['cumsum'] = cat_revenue_2017['revenue_share_2017'].cumsum()

top_80_percent_categories_2017 = cat_revenue_2017[cat_revenue_2017['cumsum'] <= 80]

top_80_percent_categories_2017




In [ ]:
#2017
cat_revenue_2017 = revenue_2017.sort_values('price', ascending=False).reset_index(drop=True)
cat_revenue_2017['cumsum'] = cat_revenue_2017['revenue_share_2017'].cumsum()

top_80_percent_categories_2017 = cat_revenue_2017[cat_revenue_2017['cumsum'] <= 80]


#2018
cat_revenue_2018 = revenue_2018.sort_values('price', ascending=False).reset_index(drop=True)
cat_revenue_2018['cumsum'] = cat_revenue_2018['revenue_share_2018'].cumsum()

top_80_percent_categories_2018 = cat_revenue_2018[cat_revenue_2018['cumsum'] <= 80]

#merging two years
revenue_share = top_80_percent_categories_2017.merge(top_80_percent_categories_2018, on='product_category_name_english', how='outer')
revenue_share.fillna(0, inplace=True)

#revenue_share['difference'] =  [round((y-x),2) if pd.notna(x) and pd.notna(y) else(round(x,2) if pd.notna(x) else round(y,2)) for x, y in zip(revenue_share['revenue_share_2017'], revenue_share['revenue_share_2018'])]
revenue_share['difference'] = (revenue_share['revenue_share_2018'] - revenue_share['revenue_share_2017']).round(2)

#revenue_share['diff_2017'] = [x if x>y else '' for x, y in zip(revenue_share['revenue_share_2017'], revenue_share['revenue_share_2018'])]
#revenue_share['diff_2018'] = [y if y>x else '' for x, y in zip(revenue_share['revenue_share_2017'], revenue_share['revenue_share_2018'])]

# Labels for 2012 bars: Show value if it exists
revenue_share['text_2017'] = revenue_share.apply(lambda row: round(row['difference'],2) if row['revenue_share_2017'] > row['revenue_share_2018']  else '', axis=1)

# Labels for 2013 bars: Show value if it exists
revenue_share['text_2018'] = revenue_share.apply(lambda row: round(row['difference'],2) if row['revenue_share_2018'] > row['revenue_share_2017'] else '', axis=1)


# Create individual Bar traces for each group
trace1 = go.Bar(name='2017', 
                x=revenue_share.query("order_year_x == 2017.0")['product_category_name_english'], 
                y=revenue_share.query("order_year_x == 2017.0")['revenue_share_2017'],
                text=revenue_share.query("order_year_x == 2017.0")['text_2017'],
                textposition='outside',
                marker_color = "#8d4ecb",
                hovertemplate="%{x}<br>Revenue share: %{y:.2f}<br>Declined: %{text}%",
                cliponaxis=False)
trace2 = go.Bar(name='2018', 
                x=revenue_share.query("order_year_y == 2018.0")['product_category_name_english'], 
                y=revenue_share.query("order_year_y == 2018.0")['revenue_share_2018'],
                text=revenue_share.query("order_year_y == 2018.0")['text_2018'],
                textposition='outside',
                marker_color = "#4ECB8D",
                hovertemplate="%{x}<br>Revenue share: %{y:.2f}<br>Growth: %{text}%",
                cliponaxis=False)

# Create the figure and add the traces
fig = go.Figure(data=[trace1, trace2])

fig.update_xaxes(title_text = "Product Categories", showgrid=False)
fig.update_yaxes(title_text = "Revenue Share (%)", showgrid=False)


fig.update_layout(title = dict(text = "<span style='font-size:20px'>Comparison of Product Categories Contributing to 80% of Revenue-(2017 vs 2018)</span>"),
                  height = 700,
                  barmode='group',
                  template = "plotly_dark",
                  hovermode = 'x unified')

# Display the chart
fig.show()




<h3>Core Revenue Drivers Stayed Consistent, but Their Weight Shifted</h3>

Categories like health_beauty, watches_gifts, bed_bath_table, sports_leisure, and computers_accessories continue to sit inside the top 80% revenue band across both years confirming their role as the platform’s revenue backbone.

However, several of these core categories experienced material shifts in revenue share, indicating changing customer demand or growing competition.

<h3>Strong Upward Momentum in Key Categories</h3>

Several categories grew their revenue share between 2017 → 2018, signalling rising demand:

<li>health_beauty (+2.63 pp) → strongest growth; category expanding rapidly and becoming a top profit driver.

<li>watches_gifts (+1.59 pp) → high-value gift purchases gaining momentum.

<li>housewares (+1.66 pp) → consistent growth in household product spending.

<li>auto (+0.75 pp) & baby (+0.99 pp) → strong upward trend, possibly tied to lifestyle and household expansion.</li>

Customers are shifting spending toward self-care, home improvement, lifestyle, and premium gifting. These categories warrant deeper inventory investment and promotional prioritization.

<h3>Decline in Some Previously Strong Categories</h3>

Certain historically large categories saw a drop in share:

<li>bed_bath_table (–0.81 pp) → still a major category, but losing relative dominance.

<li>computers_accessories (–3.14 pp) → big decline; potential price pressure, market saturation, or competition.

<li>sports_leisure (–0.13 pp) → stable but slightly weakened.

<li>toys (–2.68 pp) → demand shifting or seasonality no longer strong.</li>

These categories may require pricing reevaluation, assortment refresh, or targeted marketing to recover share.

<h3>Electronics & Office Products Show Mixed Trends</h3>

<li>computers fell (–2.59 pp) while office_furniture slightly dipped (–1.08 pp).</li>

Customers appear to be shifting spending away from traditional office/electronic items and toward lifestyle and home-related purchases.

Small Categories such as pet_shop and stationery are some of the emerging niche opportunites of 2018 which show positive share gains despite small absolute contribution and are worth exploring for future category expansion show positive share gains despite small absolute contribution.


# Sales Analysis:




In [ ]:
with open('datasets/brazil_states.json', 'r') as file:
    data = json.load(file)

sales_by_state= customer_orders.groupby(["geo_state"])["price"].sum().round().astype(int).reset_index(name="total_sales")

geo_df = gpd.GeoDataFrame.from_features(data["features"]).merge(sales_by_state, left_on="name", right_on="geo_state").set_index("geo_state")

fig = px.choropleth(geo_df,
                   geojson=geo_df.geometry,
                   locations=geo_df.index,
                   color="total_sales",
                   color_continuous_scale='Cividis',
                   hover_name=geo_df.index,
                   hover_data={'total_sales': True, 'name': True},
                   projection="mercator")
fig.update_geos(fitbounds="locations", visible=False)
fig.update_traces(hovertemplate='<b>Region:</b> %{location}<br>' +
                                '<b>Revenue:</b> %{z:$,.0f}<br><extra></extra>')
fig.update_layout(title = 'Geographical Sales Performance: Southeast Dominates Revenue', template='plotly_dark')
fig.show()

We can infer from the map above that Sao Paulo (SP) and Rio de Janeiro (RJ) had the highest e-commerce sales of any Brazilian state. I'm shocked that RJ, which is the smallest state in comparison to other Brazilian states, has outperformed other larger states in terms of sales. After doing some research, I discovered that SP and RJ are the two most populous states in Brazil. To gain a clearer picture, let's compare order value with sales.

In [ ]:
sales_by_state

orders_state = customer_orders.groupby("geo_state")["order_id"].size().sort_values(ascending=False).reset_index(name="orders")

fig = make_subplots(specs=[[{"secondary_y" : True}]])

fig.add_trace(go.Bar(x = orders_state["geo_state"], 
                     y = orders_state["orders"],
                     name = "Orders",
                     marker_color = "#8d4ecb",
                     hovertemplate = '%{y:.2s}',
                     offsetgroup = 1,
                     ),secondary_y=False)

fig.add_trace(go.Bar(x = sales_by_state["geo_state"],
                     y = sales_by_state["total_sales"],
                     name = "Sales",
                     marker_color = "#4ECB8D",
                     hovertemplate = '%{y:$,.0f}',
                     offsetgroup = 2,),secondary_y=True)

fig.update_xaxes(title = "Year and month",
                 tickangle = -60,
                 showgrid=False,)

fig.update_yaxes(title_text = "Order <b>Count</b>", secondary_y= False, showgrid=False)

fig.update_yaxes(title_text = "Sales <b>(in R$)</b>", secondary_y = True, showgrid=False)

fig.update_layout(title = "E-commerce Orders and Sales by Brazil States",
                  height = 600,
                  barmode='group',
                  font_size = 14,
                  template="plotly_dark",
                  hovermode="x unified")

fig.show()

<h3>1. São Paulo is the undisputed demand & revenue hub</h3>

Highest orders and highest sales, with demand levels more than 3× larger than the next state.

Acts as the economic engine of Brazilian e-commerce — consistent with population density & purchasing power.

<h3>2. Rio de Janeiro & Minas Gerais form the second-tier growth cluster</h3>

Strong order volume and revenue, but noticeably less than São Paulo.

These states represent high-value markets where marketing expansion or logistics investment would yield immediate returns.

<h3>3. Southern region states (RS, PR, SC) show strong per-capita performance</h3>

Although population is smaller than Southeast, their orders-to-sales ratio is stable and healthy.

Indicates mature online buying behaviour, especially in PR & RS.

<h3>4. North & Northeast have low order/sales volume</h3>

States such as Acre, Roraima, Amapá, Tocantins contribute minimally.

This gap often correlates with lower digital infrastructure, higher logistics costs and Longer delivery times which represents potential markets for targeted last-mile partnerships.

Did the Brazilian e-commerce sales mature with time?

In [ ]:
sales_by_year = customer_orders.query("order_year > 2016 & order_year_month != '2018-09'").groupby("order_year_month")["price"].sum().round().astype(int).reset_index(name="sales")

orders_by_year = customer_orders.query("order_year > 2016 & order_year_month != '2018-09'").groupby("order_year_month")["order_id"].size().reset_index(name="orders")

sales_2017 = customer_orders.query("order_year == 2017 & order_month < 9")["price"].sum()

sales_2018 = customer_orders.query("order_year == 2018 & order_year_month != '2018-09'")["price"].sum()

quarterly_sales = customer_orders.query("order_year > 2016 & order_year_month != '2018-09'").groupby('order_quarter')['price'].sum().round(decimals=0).reset_index()

freight = customer_orders.query("order_year > 2016 & order_year_month != '2018-09'").groupby("order_year_month")["freight_value"].mean().round(2).reset_index(name="avg_freight_value")

fig = make_subplots(rows=3, cols=2, row_heights=[1.5, 0.3, 0.3], column_widths=[0.5, 1.2],
                    specs=[[{"type":"xy", "secondary_y":True, "colspan":2}, None],
                           [{"type":"domain"}, {"type":"xy", "rowspan":2}],
                           [{"type":"domain"}, None]],
                           subplot_titles=["Evolution of Orders and Sales in E-commerce Over Time", None, "Revenue Growth Per Quarter (Jan2017 to Aug2018)"],
                           horizontal_spacing=0.1,
                           vertical_spacing=0.20)

# Multiple axis chart (top)
fig.add_trace(go.Bar(x = orders_by_year["order_year_month"],
                     y = orders_by_year["orders"],
                     name="orders",
                     marker=dict(color=orders_by_year["orders"], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
                     ),secondary_y=False, row=1, col=1)

fig.add_trace(go.Scatter(x = sales_by_year["order_year_month"],
                         y = sales_by_year["sales"],
                         name="Sales",
                         mode="lines+markers",
                         hovertemplate = '%{y:$,.0f}',
                         marker=dict(color = "#4ECB8D")),secondary_y=True, row=1, col=1)

# Indicators
fig.add_trace(go.Indicator(mode="number",
                           value=int(sales_2017),
                           number={"valueformat": ".3s", "prefix":"R$ ", "font":{'size': 48, 'color': '#4ECB8D'}},
                           title = {"text": "<span style='font-size:20px'>2017 Sales</span><br><span style='font-size:11px'>(Jan-Dec)</span>"}
                           ),row=2, col=1)

fig.add_trace(go.Indicator(mode="number+delta",
                           value=int(sales_2018),
                           number={"valueformat": ".3s", "prefix":"R$ ", "font":{'size': 48, 'color': '#8D4ECB'}},
                           delta={"reference": sales_2017, "relative":True, "valueformat":".0%", "position":"bottom"},
                           title = {"text": "<span style='font-size:20px'>2018 Sales</span><br><span style='font-size:11px'>(Jan-Aug)</span>"}
                           ),row=3, col=1)

# Line Chart
fig.add_trace(go.Scatter(x = quarterly_sales["order_quarter"].astype(str),
                         y = quarterly_sales["price"],
                         mode="lines+markers+text",
                         name="Revenue per quarter",
                         hovertemplate = '%{y:$,.0f}',
                         marker=dict(color = "#876FD4"),
                         text = quarterly_sales["price"],
                         texttemplate="%{y:.2s}",
                         textposition="top center",
                         textfont=dict(size=11, color='#876FD4'),
                         showlegend=False),row=2, col=2)


fig.update_xaxes(title = "Year and Month", 
                 tickmode= "array",
                 tickvals = orders_by_year["order_year_month"].unique(),
                 ticktext = orders_by_year["order_year_month"].unique(),
                 tickangle = -60,
                 showgrid=False, row=1, col=1)

fig.update_yaxes(title_text = "Orders", range=[0, 12000], secondary_y=False, showgrid=False, row=1, col=1)
fig.update_yaxes(title_text = "Sales <b>(in R$)</b>", secondary_y=True, showgrid=False, row=1, col=1)

fig.update_xaxes(title_text = "Quarter Per Year", showgrid=False, row=2, col=2)
fig.update_yaxes(title_text = "Revenue Per Quarter <b>(in R$)</b>", showgrid=False, row=2, col=2)

for ann in fig['layout']['annotations'][:7]:  # Only adjust subplot titles
    ann['y'] += 0.025

fig.update_layout(height = 1000, width = 1240,
                  legend=dict(x=0.04, y=1.0, bgcolor = 'rgba(0,0,0,0)'),
                  template = "plotly_dark",
                  hovermode = 'x unified')
fig.show()

<h3>1. Orders and revenue show strong seasonality</h3>

Growth is volume-driven, with a 7.8x rise in units sold (955 to 7,248) and a 7.5x increase in revenue (R$120,313 to R$854,686). This close match between units sold and revenue implies that growth has been predominantly driven by customer acquisition rather than price strategy.

Clear monthly uplift across 2017 → peak in Nov (Black Friday) (1M+ BRL) accounts for 18% of all 2017 sales in only one month. Because of this over-dependency, there is a significant level of business risk; if this month underperforms, it might have disastrous effects on annual performance.

Orders surge continues into early 2018, reflecting year-over-year growth and improved user onboarding.

<h3>2. Revenue nearly doubles within 18 months</h3>

<li>2017 full-year sales: R$ 3.11M

<li>2018 (Jan–Aug) sales: R$ 7.39M

<li>137% growth, even without full-year data for 2018</li>

Indicates aggressive business expansion & strong customer acquisition.

The November 2017 peak (1M+ BRL) accounts for 18% of all 2017 sales in only one month. Because of this over-dependency, there is a significant level of business risk; if this month underperforms, it might have disastrous effects on annual performance.

Over the course of five quarters from January 2017 , revenue increased fourfold from 705K to 2.8M BRL, indicating market expansion and client acquisition throughout this time period.

<li>Overall, the business is in a growth stage, not maturity.

<li>Seasonal spikes are sharp and predictable (strong planning signals).

<li>Stable performance from early 2018 suggests improved supply chain & customer acquisition.</li>

<h3>Sales & Freight Trade-offs by State:</h3>

<h4>Objective:</h4>

Evaluate how average customer spend and average freight per order differ across states relative to total sales and total freight, and surface operational and commercial drivers behind these patterns so the business can optimize pricing, product mix and logistics by state.

In [ ]:
mean_price = customer_orders.groupby("geo_state")["price"].mean().astype(int).sort_values().reset_index(name="mean_price")

fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=["Average Spend by Customer State in Brazil", "Total Sales by Customer State in Brazil"],
                    shared_yaxes='all', 
                    horizontal_spacing=0.02)

fig.add_trace(go.Bar(x = mean_price["mean_price"],
                    y = mean_price["geo_state"],
                    orientation='h',
                    name="mean price",
                    text = mean_price["mean_price"],
                    textposition= "outside",
                    hovertemplate= "%{y}<br> mean price: %{x:,0f}",
                    marker=dict(color=mean_price["mean_price"], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
                    showlegend=False),row=1, col=1)

fig.add_trace(go.Bar(x = sales_by_state["total_sales"],
                    y = sales_by_state["geo_state"],
                    orientation='h',
                    name = "total sales",
                    text = sales_by_state["total_sales"],
                    texttemplate="%{x:.2s}",
                    textposition="outside",
                    hovertemplate= "%{y}<br>Sales: %{x:.2s}",
                    marker=dict(color=sales_by_state["total_sales"], colorscale=["#89dcb3", "#4ecb8d", "#2e9e67"]),
                    showlegend=False),row=1, col=2)

fig.update_xaxes(title = "Avg Spend", showgrid=False, row=1, col=1)

fig.update_xaxes(title = "Sales", showgrid=False, row=1, col=2)

fig.update_traces(
    cliponaxis=False  # Allow text to render outside plot area
)

fig.update_layout(width = 1200, 
                  height = 700,
                  template = "plotly_dark")

fig.show()

<h3>Spend vs Revenue (high revenue, low AOV vs low revenue, high AOV)</h3>
States with very high total sales (e.g., São Paulo, RJ, MG) show low average spend per order, while low-sales states often show higher average spend per order.
<h3>Order frequency & market maturity:</h3>

<li>High-sales states are large urban markets with many repeat orders and many low-ticket everyday purchases (higher order count, lower AOV).

<li>Low-sales states may have fewer orders, but those orders contain higher-priced items (less frequent, higher AOV).</li>

<h3>Product mix differences:</h3>

<li>In big states the dominant categories could be low-price, high-frequency categories (e.g., accessories, low-ticket home goods) driving volume. Urban consumers (dense markets) behave differently — many small baskets;

<li>In remote states, purchases may be of higher-value, durable goods (customers buy rarely but spend more per purchase). Rural customers often buy only when needed and consolidate higher-value purchases.</li>

<h4>How about the freight value?</h4>

In [ ]:
mean_freight = customer_orders.groupby("geo_state")["freight_value"].mean().astype(int).sort_values().reset_index(name="mean_freight")

freight_by_state = customer_orders.groupby("geo_state")["freight_value"].sum().astype(int).reset_index(name="freight_value")

fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=["Average Freight by Customer State in Brazil", "Total Freight by Customer State in Brazil"],
                    shared_yaxes='all', 
                    horizontal_spacing=0.02)

fig.add_trace(go.Bar(x = mean_freight["mean_freight"],
                    y = mean_freight["geo_state"],
                    orientation='h',
                    name="mean freight",
                    text = mean_freight["mean_freight"],
                    textposition= "outside",
                    hovertemplate= "%{y}<br> mean freight: %{x:,0f}",
                    marker=dict(color=mean_freight["mean_freight"], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
                    showlegend=False),row=1, col=1)

fig.add_trace(go.Bar(x = freight_by_state["freight_value"],
                    y = freight_by_state["geo_state"],
                    orientation='h',
                    name = "freight value",
                    text = freight_by_state["freight_value"],
                    textposition="outside",
                    hovertemplate= "%{y}<br> freight value: %{x:,0f}",
                    marker=dict(color=freight_by_state["freight_value"], colorscale=["#89dcb3", "#4ecb8d", "#2e9e67"]),
                    showlegend=False),row=1, col=2)

fig.update_xaxes(title = "Avg Freight", showgrid=False, row=1, col=1)

fig.update_xaxes(title = "Freight Value", showgrid=False,  row=1, col=2)

fig.update_traces(
    cliponaxis=False  # Allow text to render outside plot area
)

fig.update_layout(width = 1200, 
                  height = 700, 
                  template = "plotly_dark")

fig.show()

<h3>Freight (low average freight × high total freight vs high avg freight × low total freight)</h3>

<h4>Economies of density:</h4>

High-volume states (São Paulo) benefit from short routes, dense deliveries and better carrier pricing → low avg freight per order even though total freight spend is high due to volume.

<h4>Logistics complexity for remote states:</h4>

Remote states (e.g., Roraima, Acre) have high per-order freight costs due to long distances, lower route density, and limited carrier competition → high avg freight, but total freight remains low because orders are few.

<h4>Fulfillment center proximity & routing:</h4>

Lack of regional Distribution Centers increases shipping distance/costs for remote states.

<h4>SKU size/weight differences:</h4>

If higher-AOV purchases are heavier/larger items, they will increase avg freight in that state.

Let's examine the top five Brazilian states in terms of sales and freight.

In [ ]:
#freight
top5_high_freight = customer_orders.groupby("customer_state")["freight_value"].mean().round().astype(int).nlargest(5).sort_values().reset_index(name="high_freight_value")

top5_low_freight = customer_orders.groupby("customer_state")["freight_value"].mean().round().astype(int).nsmallest(5).reset_index(name="low_freight_value")

mean_freight = customer_orders["freight_value"].mean()

#sales
top5_high_sales = customer_orders.groupby("customer_state")["price"].sum().round().astype(int).nlargest(5).sort_values().reset_index(name="highest_sales")

top5_low_sales = customer_orders.groupby("customer_state")["price"].sum().round().astype(int).nsmallest(5).reset_index(name="lowest_sales") 

avg_sales = customer_orders["price"].mean()

#products
top5_high_product = customer_orders.groupby("product_category_name_english")["price"].sum().round().astype(int).nlargest(5).sort_values().reset_index(name="high_pro_sales")

top5_low_product = customer_orders.groupby("product_category_name_english")["price"].sum().round().astype(int).nsmallest(5).reset_index(name="low_pro_sales")

top_product_sales = customer_orders.groupby("product_category_name_english")["price"].sum().max()

top_selling_product = customer_orders.groupby("product_category_name_english")["price"].sum().idxmax()

fig = make_subplots(rows=3, cols=3,
                    specs = [[{"type":"domain"}, {"type":"domain"}, {"type":"domain"}],
                             [{"type":"xy"}, {"type":"xy"}, {"type":"xy"}],
                             [{"type":"xy"}, {"type":"xy"}, {"type":"xy"}]],
                    row_heights=[0.2, 0.8, 0.8],
                    column_widths=[0.5, 0.5, 0.5],
                    subplot_titles=[None, None, None, 
                                    "<span style='font-size:14px'>Top 5 Product Category with Highest sales</span>", 
                                    "<span style='font-size:14px'>Top 5 states with highest sales</span>",
                                    "<span style='font-size:14px'>Top 5 States with<br>highest average freight value</span>", 
                                    "<span style='font-size:14px'>Top 5 Product Category with lowest sales</span>",
                                    "<span style='font-size:14px'>Top 5 States with lowest sales</span>",
                                    "<span style='font-size:14px'>Top 5 States with lowest<br>average freight value</span>",],
                    vertical_spacing=0.18, horizontal_spacing=0.1)

#Indicator-1
fig.add_trace(go.Indicator(mode="number",
                           value = int(top_product_sales.round()),
                           number={"prefix":"R$", "suffix":f"<br><span style='font-size:15px'><b>({top_selling_product})<b></span>", "font":{'size': 30}},
                           title = {"text": "<span style='font-size:18px'>Top Selling Product Category<br></span>"}),row=1,col=1)


#Indicator-2
fig.add_trace(go.Indicator(mode="number",
                           value = int(avg_sales.round()),
                           number={"prefix":"R$", "font":{'size':30}},
                           title = {"text": "<span style='font-size:18px'>Average Sales Value</span>"}),row=1,col=2)


#Indicator-3
fig.add_trace(go.Indicator(mode="number",
                           value = int(mean_freight.round()),
                           number={"prefix":"R$", "font":{'size': 30}},
                           title = {"text": "<span style='font-size:18px'>Average Freight Value</span>"}),row=1,col=3)


#Bar-1
fig.add_trace(go.Bar(x = top5_high_product["high_pro_sales"],
                     y = top5_high_product["product_category_name_english"],
                     orientation='h',
                     text=top5_high_product["high_pro_sales"],
                     texttemplate="%{x:.2s}",
                     textposition="outside",
                     cliponaxis=False,
                     hovertemplate="%{y}<br>"+ "total sales: %{x}<extra></extra>",
                     marker = dict(color=top5_high_product["high_pro_sales"], colorscale=["#89dcb3", "#4ecb8d", "#2e9e67"]),
                     showlegend=False),row=2, col=1)

fig.update_xaxes(title = dict(text="sales", font=dict(size=12)), showgrid=False, row=2, col=1)

#Bar-2
fig.add_trace(go.Bar(x = top5_high_sales["highest_sales"],
                     y = top5_high_sales["customer_state"],
                     orientation='h',
                     text=top5_high_sales["highest_sales"],
                     texttemplate="%{x:.2s}",
                     textposition="outside",
                     cliponaxis=False,
                     hovertemplate="%{y}<br>"+ "total sales: %{x}<extra></extra>",
                     marker = dict(color=top5_high_sales["highest_sales"], colorscale=["#89dcb3", "#4ecb8d", "#2e9e67"]),
                     showlegend=False),row=2, col=2)

fig.update_xaxes(title = dict(text="sales", font=dict(size=12)), showgrid=False, row=2, col=2)

#Bar-3
fig.add_trace(go.Bar(x = top5_high_freight["high_freight_value"],
                     y = top5_high_freight["customer_state"],
                     orientation='h',
                     text=top5_high_freight["high_freight_value"],
                     texttemplate="%{x:.2s}",
                     textposition="outside",
                     cliponaxis=False,
                     hovertemplate="%{y}<br>"+ "Avg freight value: %{x}<extra></extra>",
                     marker = dict(color=top5_high_freight["high_freight_value"], colorscale=["#89dcb3", "#4ecb8d", "#2e9e67"]),
                     showlegend=False),row=2, col=3)

fig.update_xaxes(title = dict(text="avg freight value", font=dict(size=12)), showgrid=False, row=2, col=3)


#Bar-4
fig.add_trace(go.Bar(x = top5_low_product["low_pro_sales"],
                     y = top5_low_product["product_category_name_english"],
                     orientation='h',
                     text=top5_low_product["low_pro_sales"],
                     texttemplate="%{x:.2s}",
                     textposition="outside",
                     cliponaxis=False,
                     hovertemplate="%{y}<br>"+ "total sales: %{x}<extra></extra>",
                     marker = dict(color=top5_low_product["low_pro_sales"], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
                     showlegend=False),row=3, col=1)

fig.update_xaxes(title = dict(text="sales", font=dict(size=12)), showgrid=False, row=3, col=1)

#Bar-5
fig.add_trace(go.Bar(x = top5_low_sales["lowest_sales"],
                     y = top5_low_sales["customer_state"],
                     orientation='h',
                     text=top5_low_sales["lowest_sales"],
                     texttemplate="%{x:.2s}",
                     textposition="outside",
                     cliponaxis=False,
                     hovertemplate="%{y}<br>"+ "total sales: %{x}<extra></extra>",
                     marker = dict(color=top5_low_sales["lowest_sales"], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
                     showlegend=False),row=3, col=2)

fig.update_xaxes(title = dict(text="sales", font=dict(size=12)), showgrid=False, row=3, col=2)

#Bar-6
fig.add_trace(go.Bar(x = top5_low_freight["low_freight_value"],
                     y = top5_low_freight["customer_state"],
                     orientation='h',
                     text=top5_low_freight["low_freight_value"],
                     texttemplate="%{x:.2s}",
                     textposition="outside",
                     cliponaxis=False,
                     hovertemplate="%{y}<br>"+ "Avg freight value: %{x}<extra></extra>",
                     marker = dict(color=top5_low_freight["low_freight_value"], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
                     showlegend=False),row=3, col=3)

fig.update_xaxes(title = dict(text="avg freight value", font=dict(size=12)), showgrid=False, row=3, col=3)

fig.update_layout(title = dict(text = "<span style='font-size:20px'><b>Comparative Analysis: E-commerce Product and Sales Performance in Brazilian States<b></span>"),
                  title_x= 0.5,
                  template = "plotly_dark",
                  height = 700)

fig.show()

# Customer Behaviour Analysis:
<h3>RFM Analysis</h3>
Objective:

To understand how different customer groups behave in terms of spending, loyalty, frequency, and value contribution — enabling better targeting, retention efforts, and revenue optimization.



In [ ]:
#Snapshot Date(date after last order):
snapshot_date = customer_orders["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

#Group by Customer to aggregate RFM metrics
rfm = customer_orders.groupby("customer_unique_id").agg(recency = ("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
                                                        frequency = ("order_id", "nunique"),
                                                        monetary = ("price", "sum")).reset_index()

#RFM Scoring and Segmentation(1 to 5 scales)
rfm["r_score"] = pd.qcut(rfm["recency"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["f_score"] = pd.cut(rfm["frequency"], bins=[0,1,2,3,4,rfm["frequency"].max()], labels=[1,2,3,4,5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], 5, labels=[1,2,3,4,5]).round().astype(int)

#Customer Segmentation according to rfm scores
def segment_customer(row):
    if row["r_score"] ==5 and row["f_score"] ==5 and row["m_score"] ==5:
        return "Champions"
    elif row["r_score"] >=4 and row["f_score"] >=4 and row["m_score"] >=4:
        return "Loyal"
    elif row["m_score"] >4 and row["r_score"] >=3:
        return "Big Spenders"
    elif row["r_score"] <=2 and row["f_score"] >=3 and row["m_score"] >=3:
        return "At Risk"
    elif row["r_score"] <=2 and row["f_score"] <=2 and row["m_score"] <=2:
        return "Hibernating"
    elif row["r_score"] ==5 and row["f_score"] ==1:
        return "New Customers"
    elif row["f_score"] ==1 and row["m_score"] <=5:
        return "One-Time Buyers"
    elif row["r_score"] >=3 and row["f_score"] >=3 and row["m_score"] <=3:
        return "Budget Shoppers"
    else:
        return "Need Attention"

rfm["customer_segment"] = rfm.apply(segment_customer, axis=1)



In [ ]:
customer_segment = rfm.groupby("customer_segment").agg(no_of_customers  = ("customer_segment", "size"),
                                                        avg_monetary = ("monetary", "mean"),
                                                        revenue = ("monetary", "sum")).round().reset_index()

customer_segment = customer_segment.assign(customer_pct = lambda x:(x["no_of_customers"]/x["no_of_customers"].sum() * 100).round(2),
                                           avg_monetary_pct = lambda x:(x["avg_monetary"]/x["avg_monetary"].sum() * 100).round(2),
                                           revenue_pct = lambda x:(x["revenue"]/x["revenue"].sum() * 100).round(2))

customer_segment



* Which segment is most profitable?
* Which segment has the most customers?

In [ ]:
headerColor = 'grey'
rowEvenColor = '#4ECB8D'
rowOddColor = '#8D4ECB'

fig = make_subplots(rows=2, cols=2, 
                    specs=[[{"type":"domain", "colspan":2}, None],
                           [{"type":"xy"},{"type":"xy"}]],
                    subplot_titles=["Distribution of E-commerce Revenue and Customers by Customer Segment", "Average Spend by Customer Segment(in R$)", "Total Spend By Customer Segment(in R$)"],
                    vertical_spacing=0.1, horizontal_spacing=0.15)

#Table
fig.add_trace(go.Table(
  columnwidth=[1.3, 1.2, 1, 1, 1, 1.2, 1],
  header=dict(
    values=list(customer_segment.columns),
    line_color='darkslategray',
    fill_color=headerColor,
    align=['left','center'],
    font=dict(color='white', size=15)
  ),
  cells=dict(
    values=[customer_segment[col].to_list() for col in customer_segment.columns],
    line_color='darkslategray',
    # 2-D list of colors for alternating rows
    fill_color = [[rowOddColor,rowEvenColor,rowOddColor, rowEvenColor,rowOddColor,rowEvenColor,rowOddColor, rowEvenColor,rowOddColor]*9],
    align = ['left', 'center'],
    height = 27,
    font = dict(color = 'white', size = 14)
    )),row=1,col=1)

#Bar-1

df_sorted = customer_segment.sort_values(by='avg_monetary', ascending=True)
fig.add_trace(go.Bar(x = df_sorted["avg_monetary"],
                     y = df_sorted["customer_segment"],
                     orientation = 'h',
                     hovertemplate = "%{y}<br>" + "Avg Price: %{x}<extra></extra>",
                     text = df_sorted["avg_monetary"],
                     textposition="outside",
                     marker = dict(color=df_sorted["avg_monetary"], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
                     cliponaxis=False,
                     showlegend = False),row=2, col=1)

fig.update_yaxes(title = "", row=2,col=1)
fig.update_xaxes(title = "Avg_Price",showgrid=False, row=2, col=1)

#Bar-2
fig.add_trace(go.Bar(x = df_sorted["revenue"],
                    y = df_sorted["customer_segment"],
                    orientation = 'h',
                    hovertemplate = "%{y}<br>" + "Total Revenue: %{x:.2s}<extra></extra>",
                    marker = dict(color=df_sorted["revenue"], colorscale=["#89dcb3", "#4ecb8d", "#2e9e67"]),
                    text = df_sorted["revenue"].map('{:.0f}'.format),
                    textposition="outside",
                    texttemplate="%{x:.2s}",
                    cliponaxis=False,
                    showlegend = False),row=2, col=2)

fig.update_xaxes(title = "Total_Revenue", showgrid=False,  row=2,col=2)

fig.update_layout(height=800, 
                  template="plotly_dark")
fig.show()




The table reveals a strong imbalance in value contribution: a small % of customers drives disproportionately large revenue, while the majority contributes low spend and low retention.

<h3>Key Pattern-Driven Insights</h3>
<h4>1. “One-Time Buyers” dominate customer count but not value</h4>

<li>They make up 53% of customers yet contribute only 50% of total revenue.

<li>Their average spend is very low (R$135), meaning they inflate acquisition cost but not long-term value.</li>

The company is burning acquisition budget on people who never return.
Solution: Introduce first-purchase incentives, follow-up campaigns, and cross-sell nudges to convert them into recurring buyers.

<h4>2. “Big Spenders” are the revenue backbone despite being only 12% of customers</h4>

<li>They contribute 33% of all revenue, with 3× higher average spend than most segments.

<li>But their count is shrinking toward the end of the timeline (seen in the next visual).</li>

Losing them even slightly has a massive revenue impact.
Solution: Immediate retention program (VIP perks, faster delivery, loyalty tiers, personalized emails).

<h4>3. “Champions” spend the most per purchase but are extremely few</h4>

<li>Only 8 customers, yet the highest average order value (R$755).

<li>Their overall revenue share is tiny simply because there are so few of them.</li>

This segment has huge revenue upside if expanded.
Solution: Look at what categories/products they purchase → build look-alike audience → target with premium bundles.

<h4>4. “Hibernating” customers (17% of base) show the highest churn risk</h4>

<li>Low purchase frequency and long inactivity periods.

<li>But revenue contribution is not negligible (5%).</li>

They are recoverable and cheaper to re-activate than new customers.
Solution: Dormant-customer win-back program (discount code, “We miss you” reminders).

<h4>5. “Need Attention” buyers are the most promising upgrade segment</h4>

<li>Medium frequency + moderate spend.

<li>They behave like “almost loyal” customers but aren’t fully retained yet.</li>

This is the highest-ROI group for up-selling.
Solution: Personalized recommendation emails, targeted promotions on categories they frequently explore.

# Customer Targeting:
Based on the insights gathered from the customer segment study we will focus on three customer segments for sutomer retention and potential future revenue oppurtunities:

* One-Time Buyers (53% of customers and 50% of revenue)

        -> Problem : High customer churn(one purchase, no loyalty), even though they are high volume customers they are One-Time Buyers so it is necessary to engage with them and reactivate them to retain the customers and thier enormous revenue volume.
* Big Spenders (12% of Customers, 33% of Revenue)

        -> Despite the fact that 50% of revenue comes from One-Time Buyers which is unsustainable, so in order to boost and sustain revenue we need to concentrate on Big Spenders as the small group of customers driving 1/3 of revenue it will make a huge difference if we attempt to persuade them to become loyal customers.
* New Customers (16% of Customers, 9% of Revenue)

        -> New customers are the third in most profitable and big customer base we need to nurture new customers otherwise new customers could become One-Time Buyers.


<h3>Targeted Customer Segments (One-Time Buyers, Big Spenders, New Customers)</h3>

In [ ]:
one_time_buyers = customer_orders[customer_orders["customer_unique_id"].isin(rfm[rfm["customer_segment"] == 'One-Time Buyers']["customer_unique_id"])]

big_spenders = customer_orders[customer_orders['customer_unique_id'].isin(rfm[rfm['customer_segment'] == 'Big Spenders']['customer_unique_id'])]

new_customers = customer_orders[customer_orders['customer_unique_id'].isin(rfm[rfm['customer_segment'] == 'New Customers']['customer_unique_id'])]



In [ ]:
#Bar-line plot data
one_time_buyers_data = one_time_buyers.query("order_year > 2016 & order_year_month != '2018-09'")\
                                      .groupby("order_year_month").agg(count = ('order_id', 'count'),
                                                                        sales = ('price', 'sum')).reset_index()

big_spenders_data = big_spenders.query("order_year > 2016 & order_year_month != '2018-09'")\
                                .groupby("order_year_month").agg(count = ('order_id', 'count'),
                                                                 sales = ('price', 'sum')).reset_index()

new_customers_data = new_customers.query("order_year > 2016 & order_year_month != '2018-09'")\
                                  .groupby("order_year_month").agg(count = ('order_id', 'count'),
                                                                   sales = ('price', 'sum')).reset_index()
#Indicators Data
otb_interval = one_time_buyers.sort_values(by="order_purchase_timestamp").groupby("customer_unique_id")["order_purchase_timestamp"].apply(lambda x: x.diff().mean()).dt.days.mean()

bs_interval = big_spenders.sort_values(by="order_purchase_timestamp").groupby('customer_unique_id')['order_purchase_timestamp'].apply(lambda x: x.diff().mean()).dt.days.mean()

nc_interval = new_customers.sort_values(by="order_purchase_timestamp").groupby('customer_unique_id')['order_purchase_timestamp'].apply(lambda x: x.diff().mean()).dt.days.mean()

otb_max = one_time_buyers.sort_values(by="order_purchase_timestamp").groupby("customer_unique_id")["order_purchase_timestamp"].apply(lambda x: x.diff().mean()).dt.days.max()

bs_max = big_spenders.sort_values(by="order_purchase_timestamp").groupby('customer_unique_id')['order_purchase_timestamp'].apply(lambda x: x.diff().mean()).dt.days.max()

nc_max = new_customers.sort_values(by="order_purchase_timestamp").groupby('customer_unique_id')['order_purchase_timestamp'].apply(lambda x: x.diff().mean()).dt.days.max()

#Grouped Bar Data(products)
otb_top3_product = one_time_buyers.groupby("product_category_name_english").agg(count = ('order_id', 'count'),
                                                                                sales = ('price', 'sum'))\
                                                                            .nlargest(columns=['count', 'sales'], n=3).reset_index()

bs_top3_product = big_spenders.groupby("product_category_name_english").agg(count = ('order_id', 'count'),
                                                                                sales = ('price', 'sum'))\
                                                                            .nlargest(columns=['count', 'sales'], n=3).reset_index()

nc_top3_product = new_customers.groupby("product_category_name_english").agg(count = ('order_id', 'count'),
                                                                                sales = ('price', 'sum'))\
                                                                            .nlargest(columns=['count', 'sales'], n=3).reset_index()

#Grouped bar chart (Reviews)
otb_reviews = one_time_buyers.groupby("review_score").agg(count = ("review_score", "count"),
                                            avg_delivery_days = ("delivery_days", "mean")).reset_index()

bs_reviews = big_spenders.groupby("review_score").agg(count = ("review_score", "count"),
                                            avg_delivery_days = ("delivery_days", "mean")).reset_index()

nc_reviews = new_customers.groupby("review_score").agg(count = ("review_score", "count"),
                                            avg_delivery_days = ("delivery_days", "mean")).reset_index()


fig = make_subplots(rows=4, cols=3, 
                    specs=[[{"type":"xy" ,"secondary_y" : True, 'colspan':3}, None, None],
                           [{"type":"domain"}, {"type":"domain"}, {"type":"domain"}],
                           [{"type":"xy", "secondary_y": True}, {"type":"xy", "secondary_y": True}, {"type":"xy", "secondary_y": True}],
                           [{"type":"xy", "secondary_y": True}, {"type":"xy", "secondary_y": True}, {"type":"xy", "secondary_y": True}]],
                    row_heights=[1.5, 0.2, 0.8, 0.8], 
                    subplot_titles=["<span style='font-size:18px'>Order and Sales Distribution by Three Targeted Customer Segments</span>",None, None, None,
                                    "<span style='font-size:14px'>Top 3 Product Category of One-Time Buyers</span>", 
                                    "<span style='font-size:14px'>Top 3 Product Category of Big Spenders</span>",
                                    "<span style='font-size:14px'>Top 3 Product Category of New Customers</span>", 
                                    "<span style='font-size:14px'>Distribution of One-Time Buyers Reviews</span>",
                                    "<span style='font-size:14px'>Distribution of Big Spenders Reviews</span>",
                                    "<span style='font-size:14px'>Distribution of New Customers Reviews</span>",],
                    horizontal_spacing=0.15)

fig.add_trace(go.Bar(x = one_time_buyers_data["order_year_month"],
                     y = one_time_buyers_data["count"],
                     name = "One-Time Buyers",
                     marker_color = "#4ecb8d",
                     offsetgroup = 1,
                     ),secondary_y=False, row=1, col=1)

fig.add_trace(go.Bar(x = big_spenders_data["order_year_month"],
                     y = big_spenders_data["count"],
                     name = "Big Spenders",
                     marker_color = "#8d4ecb",
                     offsetgroup = 2,
                     ),secondary_y=False, row=1, col=1)

fig.add_trace(go.Bar(x = new_customers_data["order_year_month"],
                     y = new_customers_data["count"],
                     name = "New Customers",
                     marker_color = "#01cdfe",
                     offsetgroup = 3,
                     ),secondary_y=False, row=1, col=1)

fig.add_trace(go.Scatter(x = one_time_buyers_data["order_year_month"],
                         y = one_time_buyers_data["sales"],
                         name="One-Time Buyers",
                         mode="lines+markers",
                         hovertemplate = '%{y:$,.0f}',
                         marker=dict(color = "#4ECB8D"),
                         showlegend=False),secondary_y=True, row=1, col=1)

fig.add_trace(go.Scatter(x = big_spenders_data["order_year_month"],
                         y = big_spenders_data["sales"],
                         name="Big Spenders",
                         mode="lines+markers",
                         hovertemplate = '%{y:$,.0f}',
                         marker=dict(color = "#8d4ecb"),
                         showlegend=False),secondary_y=True, row=1, col=1)

fig.add_trace(go.Scatter(x = new_customers_data["order_year_month"],
                         y = new_customers_data["sales"],
                         name="New Customers",
                         mode="lines+markers",
                         hovertemplate = '%{y:$,.0f}',
                         marker=dict(color = "#01cdfe"),
                         showlegend=False),secondary_y=True, row=1, col=1)

fig.update_xaxes(title = "Year and Month", 
                 tickmode= "array",
                 tickvals = big_spenders_data["order_year_month"].unique(),
                 ticktext = big_spenders_data["order_year_month"].unique(),
                 tickangle = -60, row=1, col=1)

fig.update_yaxes(title = "Order Count", secondary_y=False, showgrid=False, row=1, col=1)
fig.update_yaxes(title = "Sales in (R$)", secondary_y=True, showgrid=False, row=1, col=1)

#second row(Indicators)
fig.add_trace(go.Indicator(mode = "number",
                           value=int(otb_interval.round()),
                           #delta = {"reference": bs_max, "position":"bottom"},
                           number={"suffix": f"<span style='font-size:20px; color:#4ecb8d;'> days</span> <br> <span style='font-size:20px; color:red;'>max: ▼ {int(otb_max)} days</span>", "font":{'size': 40, 'color': '#4ECB8D'}},
                           title={"text": "<span style='font-size:20px'>One-Time Buyers</span>"}),row=2,col=1)

fig.add_annotation(
    text="is the average interval between orders",
    xref="paper", yref="paper",
    x=0, y=0.55,  # Adjust coordinates as needed
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="center")

#2
fig.add_trace(go.Indicator(mode = "number",
                           value=int(bs_interval.round()),
                           number={"suffix": f"<span style='font-size:20px; color:#4ecb8d;'> days</span> <br> <span style='font-size:20px; color:red;'>max: ▼ {int(bs_max)} days</span>", "font":{'size': 40, 'color': '#4ECB8D'}},
                           title={"text": "<span style='font-size:20px'>Big Spenders</span>"}),row=2,col=2)

fig.add_annotation(
    text="is the average interval between orders",
    xref="paper", yref="paper",
    x=0.46, y=0.55,  # Adjust coordinates as needed
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="center")

#3
fig.add_trace(go.Indicator(mode = "number",
                           value=int(nc_interval.round()),
                           number={"suffix": f"<span style='font-size:20px; color:#4ecb8d;'> days</span> <br> <span style='font-size:20px; color:red;'>max: ▼ {int(nc_max)} days</span>", "font":{'size': 40, 'color': '#4ECB8D'}},
                           title={"text": "<span style='font-size:20px'>New Customers</span>"}),row=2,col=3)

fig.add_annotation(
    text="is the average interval between orders",
    xref="paper", yref="paper",
    x=0.95, y=0.55,  # Adjust coordinates as needed
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="center")

#third row(grouped bar)
fig.add_trace(go.Bar(x=otb_top3_product["product_category_name_english"],
                     y=otb_top3_product["count"],
                     width=[0.3, 0.3, 0.3],
                     name = "Orders",
                     marker = dict(color = "#4ECB8D"),
                     hovertemplate = '%{x}<br>Orders: %{y}<extra></extra>',
                     showlegend=False), secondary_y=False, row=3,col=1)

fig.add_trace(go.Scatter(x = otb_top3_product["product_category_name_english"],
                     y = otb_top3_product["sales"],
                     name = "Sales",
                     mode="lines+markers",
                     marker = dict(color= "#4ECB8D"),
                     hovertemplate = 'Sales: %{y:$,.0f}<extra></extra>',
                     showlegend=False),secondary_y=True, row=3, col=1)

fig.update_xaxes(title = dict(text="Product Category", font=dict(size=12)), tickangle = -20, row=3, col=1)
fig.update_yaxes(title = dict(text="Orders Count", font=dict(size=12)), secondary_y=False, showgrid=False, row=3, col=1)
fig.update_yaxes(title = dict(text="Sales in (R$)", font=dict(size=12)), secondary_y= True, showgrid=False, row=3, col=1)

#2
fig.add_trace(go.Bar(x=bs_top3_product["product_category_name_english"],
                     y=bs_top3_product["count"],
                     width = [0.3, 0.3, 0.3],
                     name = "Orders",
                     marker = dict(color = "#8d4ecb"),
                     hovertemplate = '%{x}<br>Orders: %{y}<extra></extra>',
                     showlegend=False), secondary_y=False, row=3,col=2)

fig.add_trace(go.Scatter(x = bs_top3_product["product_category_name_english"],
                     y = bs_top3_product["sales"],
                     name = "Sales",
                     mode="lines+markers",
                     marker = dict(color= "#8d4ecb"),
                     hovertemplate = 'Sales: %{y:$,.0f}<extra></extra>',
                     showlegend=False),secondary_y=True, row=3, col=2)

fig.update_xaxes(title = dict(text="Product Category", font=dict(size=12)), tickangle = -20, row=3, col=2)
fig.update_yaxes(title = dict(text="Orders Count", font=dict(size=12)), secondary_y=False, showgrid=False, row=3, col=2)
fig.update_yaxes(title = dict(text="Sales in (R$)", font=dict(size=12)), secondary_y= True, showgrid=False, row=3, col=2)

#3
fig.add_trace(go.Bar(x=nc_top3_product["product_category_name_english"],
                     y=nc_top3_product["count"],
                     width = [0.3, 0.3, 0.3],
                     name = "Orders",
                     marker = dict(color = "#01cdfe"),
                     hovertemplate = '%{x}<br>Orders: %{y}<extra></extra>',
                     showlegend=False), secondary_y=False, row=3,col=3)

fig.add_trace(go.Scatter(x = nc_top3_product["product_category_name_english"],
                     y = nc_top3_product["sales"],
                     name = "Sales",
                     mode="lines+markers",
                     marker = dict(color= "#01cdfe"),
                     hovertemplate = 'Sales: %{y:$,.0f}<extra></extra>',
                     showlegend=False),secondary_y=True, row=3, col=3)

fig.update_xaxes(title = dict(text="Product Category", font=dict(size=12)), tickangle = -20, row=3, col=3)
fig.update_yaxes(title = dict(text="Orders Count", font=dict(size=12)), secondary_y=False, showgrid=False, row=3, col=3)
fig.update_yaxes(title = dict(text="Sales in (R$)", font=dict(size=12)), secondary_y= True, showgrid=False, row=3, col=3)

#Fourth row(grouped bar chart)
fig.add_trace(go.Bar(x=otb_reviews["review_score"],
                     y=otb_reviews["count"],
                     width=[0.3, 0.3, 0.3, 0.3, 0.3],
                     name = "Reviews",
                     marker = dict(color = "#4ECB8D"),
                     hovertemplate = '%{x}<br>Reviews: %{y}<extra></extra>',
                     showlegend=False), secondary_y=False, row=4,col=1)

fig.add_trace(go.Scatter(x = otb_reviews["review_score"],
                     y = otb_reviews["avg_delivery_days"],
                     name = "Avg delivery days",
                     mode="lines+markers",
                     marker = dict(color= "#4ECB8D"),
                     hovertemplate = 'Avg delivery days: %{y:,.0f}<extra></extra>',
                     showlegend=False),secondary_y=True, row=4, col=1)

fig.update_xaxes(title = dict(text="Review Score", font=dict(size=12)),
                tickmode= "array",
                tickvals = otb_reviews["review_score"].unique(),
                ticktext = otb_reviews["review_score"].unique(), row=4, col=1)
fig.update_yaxes(title = dict(text="Review Count", font=dict(size=12)), secondary_y=False, showgrid=False, row=4, col=1)
fig.update_yaxes(title = dict(text="Avg Delivery Day", font=dict(size=12)), secondary_y= True, showgrid=False, row=4, col=1)

#2
fig.add_trace(go.Bar(x = bs_reviews["review_score"],
                     y = bs_reviews["count"],
                     width = [0.3, 0.3, 0.3, 0.3, 0.3],
                     name = "Reviews",
                     marker = dict(color = "#8d4ecb"),
                     hovertemplate = '%{x}<br>Reviews: %{y}<extra></extra>',
                     showlegend=False), secondary_y=False, row=4,col=2)

fig.add_trace(go.Scatter(x = bs_reviews["review_score"],
                     y = bs_reviews["avg_delivery_days"],
                     name = "Avg delivery days",
                     mode="lines+markers",
                     marker = dict(color= "#8d4ecb"),
                     hovertemplate = 'Avg delivery days: %{y:,.0f}<extra></extra>',
                     showlegend=False),secondary_y=True, row=4, col=2)

fig.update_xaxes(title = dict(text="Review Score", font=dict(size=12)),
                tickmode= "array",
                tickvals = otb_reviews["review_score"].unique(),
                ticktext = otb_reviews["review_score"].unique(), row=4, col=2)
fig.update_yaxes(title = dict(text="Review Count", font=dict(size=12)), secondary_y=False, showgrid=False, row=4, col=2)
fig.update_yaxes(title = dict(text="Avg Delivery Day", font=dict(size=12)), secondary_y= True, showgrid=False, row=4, col=2)

#3
fig.add_trace(go.Bar(x=nc_reviews["review_score"],
                     y=nc_reviews["count"],
                     width = [0.3, 0.3, 0.3, 0.3, 0.3],
                     name = "Reviews",
                     marker = dict(color = "#01cdfe"),
                     hovertemplate = '%{x}<br>Reviews: %{y}<extra></extra>',
                     showlegend=False), secondary_y=False, row=4,col=3)

fig.add_trace(go.Scatter(x = nc_reviews["review_score"],
                     y = nc_reviews["avg_delivery_days"],
                     name = "Avg delivery days",
                     mode="lines+markers",
                     marker = dict(color= "#01cdfe"),
                     hovertemplate = 'Avg delivery days: %{y:,.0f}<extra></extra>',
                     showlegend=False),secondary_y=True, row=4, col=3)

fig.update_xaxes(title = dict(text="Review Score", font=dict(size=12)),
                tickmode= "array",
                tickvals = otb_reviews["review_score"].unique(),
                ticktext = otb_reviews["review_score"].unique(), row=4, col=3)
fig.update_yaxes(title = dict(text="Review Count", font=dict(size=12)), secondary_y=False, showgrid=False, row=4, col=3)
fig.update_yaxes(title = dict(text="Avg Delivery Day", font=dict(size=12)), secondary_y= True, showgrid=False, row=4, col=3)

for ann in fig['layout']['annotations'][:7]:  # Only adjust subplot titles
    ann['y'] += 0.025

fig.update_layout(height = 1500,
                  legend = dict(orientation='h', yanchor = 'top', y=1.09, xanchor = 'right', x=0.95),
                  template="plotly_dark",
                  hovermode = 'x')
fig.show()


<h3>Ordering/ Sales Patterns Over Time</h3>
<h4>One-Time Buyers:</h4>
<li>Spike in Nov 2017: Indicates One-Time Buyers are seasobal buyers coinciding with Black-Friday Sales or Holiday Sales.

<li>Post-Spike Stagnation: Revenue flatlined despite order growth → Price cuts or low-value purchases.</li>

This suggests that a large share of the company’s customer base is holiday-driven, discount-driven, or impulse-driven, these people come for seasonal deals and leave.

Opportunity: Build re-engagement flows right after peak months.

<h4>Big Spenders:</h4>
<li>Late 2017 Takeoff: Their sales curve is smoother and less seasonal and post holiday sales indicating they are need-based, value-driven shoppers — not impulse buyers.
<li>Fluctuating Sales according to Orders: Medium Average Order Value(AOV) but inconsistent engagement, their order frequency is low (avg 30 days apart) but steady.</li>

Opportunity: Recommend premium bundles, extended warranties, high-ticket products.

<h4>New Customers:</h4>
Fluctuating Sales according to Orders: Even though the orders are high the sales slightly increases and dips according to the orders it shows that that the avgerage spend of new customers is low that's the reason the sales remain stagnant depsite the increasing order value(lookup the previous chart to know the average spend of each customer segments)

<h3>Customer Lifetime Intervals</h3>
<li>One-Time Buyers and New Customers have zero-day intervals</li>

They buy once and never come back, the problem is not acquisition but retention.

<li>Big Spenders have extremely long max intervals (572 days)</li>

They disappear for long stretches yet return, they are not loyal, they are high value but inconsistent.

They are vulnerable to competitor influence.

Opportunity: Send product reminders every 20–30 days, exclusive member-only benefits.

<h3>Category Preferences</h3>
Every segment gravitates toward different product categories

<li>One-Time Buyers → low-risk essentials (bed_bath_table, health_beauty, sports).

<li>Big Spenders → premium categories (watches_gifts, health_beauty, furniture-related décor).

<li>New Customers → household essentials (housewares, bed_bath_table).</li>

It's obvious that the top three product categories across all three customer segments were health_beauty, and bed_bath_table, which are the two most lucrative product categories. Additionally, the watches_gifts category, which includes luxury goods, is the second most popular sector where Big Spenders want to spend their money.

Opportunity: Target each group with category-based recommendations.

<h3>Review Score Behavior</h3>
Clear correlation: Lower-rated customers receive slower delivery whereas Higher-rated customers receive faster delivery

Across all three groups:

<li>Lower review scores correlate with longer delivery times.

<li>High review scores correlate with faster delivery.</li>

Logistic performance directly affects customer satisfaction — not product quality.

Opportunity: Improve delivery SLAs for high-value and at-risk segments to prevent churn.



<h3>Strategic Recommendations</h3>

<h3>1. One-Time Buyers: </h3>
<li>Holiday Prep: Replicate Nov 2017’s strategy with early-Black Friday deals.
<li>Post-Purchase Nudge: “Order again within 48 hours for free shipping.” (By this we can persuade One-Time Buyers to buy more than one time)</li>

<h3>2. Big Spenders:</h3>
<li>Frequent Buyers(Order Interval less than 90 days): Launch a subscription model (e.g., “Monthly luxury skincare box”).
<li>Seasonal Buyers: Target with pre-holiday exclusive previews.</li>

<h3>3. New Customers:</h3>
<li>Same-day Purchase Optimization : New customers who made more than one orders made their second purchase at the same day and same time (15%) maybe they're looking for other products to buy at once so the best strategy is to use bundle prompts on top 3 products.</li>
<ul><li>eg: “Customers who bought [health_beauty] also added [bed_bath_table] — save 10% on bundles!”</li></ul>   

<h3>4. Delivery Crisis:</h3>
<li>Top 3 Slow Categories: Negotiate with suppliers for faster shipping.</li>

# Cohort analysis:
Objective

To understand how well the business retains customers over time and identify structural issues in customer loyalty, post-purchase experience, and repeat purchase behavior.

In Cohort Analysis we group customers by their first purchase month and tracks how many of them return over time, It enables us to recognize customer retention behaviour and identify patterns in customer loyalty or drop-off.

In [ ]:
customer_orders["cohort_month"] = customer_orders.groupby("customer_unique_id")["order_purchase_timestamp"].transform("min").dt.to_period('M')

def get_month_diff(df):
    return (df["order_year"] - df["cohort_month"].year) * 12 + (df["order_month"] - df["cohort_month"].month)

customer_orders["cohort_index"] = customer_orders.apply(get_month_diff, axis=1)



excluded_values = [pd.Period('2016-09', freq='M'), pd.Period('2016-10', freq='M'), pd.Period('2016-12', freq='M')]
customer_orders[~customer_orders["cohort_month"].isin(excluded_values)]

cohort_data = customer_orders[~customer_orders["cohort_month"].isin(excluded_values)].groupby(["cohort_month","cohort_index"])["customer_unique_id"].nunique().reset_index()

cohort_pivot = cohort_data.pivot(index="cohort_month", columns="cohort_index", values="customer_unique_id")

cohort_sizes = cohort_pivot.iloc[:, 0]

customer_retention_rate = cohort_pivot.divide(cohort_sizes, axis=0)



excluded_values = [pd.Period('2016-09', freq='M'), pd.Period('2016-10', freq='M'), pd.Period('2016-12', freq='M')]
customer_orders[~customer_orders["cohort_month"].isin(excluded_values)]

In [ ]:
# Convert index/columns to strings so they're JSON serializable
z = customer_retention_rate.values
x = customer_retention_rate.columns.astype(str)
y = customer_retention_rate.index.astype(str)

z_display = np.where(np.isnan(z), 0.0, z)

fig = go.Figure(data=go.Heatmap(z = z,
                                x = x,
                                y = y,
                                colorscale="Viridis",
                                zmin=0,
                                zmax=1,
                                colorbar=dict(title="Retention Rate"),
                                customdata = z_display,
                                hovertemplate="Cohort Month: %{y}<br>Month Index: %{x}<br>Retention: %{z:.1%}<extra></extra>"))

for i in range(len(y)):
    for j in range(len(x)):
        val = z[i][j]
        if not np.isnan(val):  # Avoid placing text on missing data
            fig.add_annotation(
                text=f"{val:.1%}",  # Format as percentage
                x=x[j],
                y=y[i],
                showarrow=False,
                font=dict(color="white" if val < 0.5 else "black", size=12),
                xanchor="center",
                yanchor="middle"
            )
fig.update_xaxes(title = "Cohort Index (Months Since First Purchase)", side="top", showgrid = False)
fig.update_yaxes(title = "Cohort Month",
                 autorange="reversed",
                 tickmode = "array",
                 tickvals = y.unique(),
                 ticktext = y.unique(), 
                 showgrid = False)

fig.update_layout(height = 900,
    title=dict(
        text="Monthly Customer Retention Heatmap",
        x=0.48,
        y = 0.98,
        font=dict(size=20)
    ),
    #plot_bgcolor="white"
    template="plotly_dark")

fig.show()



Cohort analysis reveals that the platform suffers from near-zero retention across all customer cohorts, with less than 1% returning after the first purchase. This pattern is consistent across 20+ months and indicates a structural dependence on acquisition rather than loyalty. To build long-term revenue stability, the business must focus on retention strategies, delivery improvements, and expanding into higher-frequency product categories.”

<h3>Retention collapses immediately after the first purchase (0.1%–0.7%)</h3>

Every cohort begins with 100% at month zero (the purchase month), but retention drops almost instantly to below 1% from Month 1 onward.

This means:

<li>Customers rarely come back after the first order indicates customers treat the platform as a low-frequency necessity

<li>Repeat purchase behavior is almost nonexistent

<li>The business is overwhelmingly dependent on constant acquisition</li>

<h3>Peaks at 0.7% retention show micro-loyalty pockets tied to logistics</h3>

Small spikes at Months 8, 9, 10 and 11 appear for some cohorts (0.6%–0.7%).

<li>This is NOT random maybe due to Black Friday and Holiday Sales or repeat purchase categories expansion and pricing

<li>Earlier visuals (delivery vs review) showed: Faster deliveries → higher review scores → higher chance of reordering</li>

This means the micro-retained customers likely received good delivery service and Improving delivery consistency could multiply retention.

# Average Order Value (AOV) Analysis 
<h3>Objective</h3>

To analyze how Average Order Value (AOV) behaves over time and across product categories, uncovering which categories generate high-value baskets, and identifying macro patterns linking AOV, sales volume, and customer purchasing behavior.

In [ ]:
df_aov = customer_orders.query("order_year > 2016 & order_year_month != '2018-09'")[['order_id','order_year_month','price']].reset_index().drop(columns='index')

df_aov['order_year_month'] = pd.to_datetime(df_aov['order_year_month'])

monthly = (
    df_aov.groupby('order_year_month')
    .agg(
        sales=('price', 'sum'),
        orders=('order_id', 'nunique')
    )
    .reset_index()
)

full_range = pd.date_range(
    start=monthly['order_year_month'].min(),
    end='2018-12',
    freq='MS'       
)
full_range = full_range.strftime('%Y-%m')
monthly['order_year_month'] = monthly['order_year_month'].dt.strftime('%Y-%m')

full_date_df = pd.DataFrame({'order_year_month': full_range})

monthly_aov = full_date_df.merge(
    monthly,
    on='order_year_month',
    how='left'
)

monthly_aov['aov'] = (monthly_aov['sales'] / monthly_aov['orders']).round()

monthly_aov = monthly_aov.set_index('order_year_month')

train = monthly_aov.loc[:'2018-08']
future = monthly_aov['2018-09':] 

# SALES MODEL
model_sales = SARIMAX(
    train['sales'],
    order=(1,1,1),
    seasonal_order=(1,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sales_fit = model_sales.fit()

sales_forecast = sales_fit.predict(start='2018-09', end='2018-12')

monthly_aov.loc['2018-09':'2018-12', 'sales'] = sales_forecast

# Orders MODEL
model_sales = SARIMAX(
    train['orders'],
    order=(1,1,1),
    seasonal_order=(1,0,0,12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

orders_fit = model_sales.fit()

orders_forecast = orders_fit.predict(start='2018-09', end='2018-12')

monthly_aov.loc['2018-09':'2018-12', 'orders'] = orders_forecast

# AOV Model
model_aov = SARIMAX(
    train['aov'],
    order=(1,1,1),
    seasonal_order=(1,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

aov_fit = model_aov.fit()
aov_forecast = aov_fit.predict(start='2018-09', end='2018-12')
monthly_aov.loc['2018-09':'2018-12', 'aov'] = aov_forecast

pd.set_option('display.float_format', lambda x: f'{x:,.0f}')
monthly_aov = monthly_aov.reset_index()
monthly_aov.loc[:19, 'type'] = "Actual"
monthly_aov.loc[20:, 'type'] = "Forecast"


monthly_aov


In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

bar_colors = ["#8D4ECB" if forecast == 'Actual' else '#B4A3D9' for forecast in monthly_aov['type']]
line_colors = ["#4ECB8D" if forecast == 'Actual' else '#C2F8CB' for forecast in monthly_aov['type']]
fig.add_trace(go.Bar(
    x=monthly_aov['order_year_month'],
    y=monthly_aov['sales'],
    name="Sales",
    marker = dict(color = bar_colors),
    text=monthly_aov['sales'],
    textposition="outside",
    texttemplate="%{y:.2s}",
    hovertemplate = 'Sales: %{y:.2s}<extra></extra>'
    ), secondary_y=False)

fig.update_xaxes(title = 'Year and Month',
                 tickmode= "array",
                 tickvals = monthly_aov['order_year_month'].unique(),
                 ticktext = monthly_aov['order_year_month'].unique(),
                 tickangle = -60)

fig.update_yaxes(title = 'Sales in (R$)', showgrid=False, secondary_y=False)
fig.update_yaxes(title = "Average Order Value", showgrid=False, secondary_y=True)

fig.add_trace(go.Scatter(
    x = monthly_aov['order_year_month'],
    y = monthly_aov['aov'],
    name = "AOV",
    mode="lines+markers",
    line = dict(color= "#4ECB8D"),
    marker=dict(color=line_colors),
    hovertemplate = 'AOV: %{y:.0f}<extra></extra>'
    ),secondary_y=True)

fig.update_layout(title="<span style='font-size:16px'>Monthly Sales and Average Order Value (BRL) - 2017 to 2018</span><br><span style='font-size:14px'>(with forecast for Q4 2018)</span>",
                  template="plotly_dark",
                  height=600,
                  hovermode='x unified')

fig.show()

In [ ]:
import plotly.io as pio
fig = make_subplots(specs=[[{"secondary_y": True}]])

bar_colors = ["#8D4ECB" if forecast == 'Actual' else '#B4A3D9' for forecast in monthly_aov['type']]
line_colors = ["#4ECB8D" if forecast == 'Actual' else '#C2F8CB' for forecast in monthly_aov['type']]
fig.add_trace(go.Bar(
    x=monthly_aov['order_year_month'],
    y=monthly_aov['orders'],
    name="Orders",
    marker = dict(color = bar_colors),
    hovertemplate = 'Orders: %{y:.0f}<extra></extra>'
    ), secondary_y=False)

fig.update_xaxes(title = 'Year and Month',
                 tickmode= "array",
                 tickvals = monthly_aov['order_year_month'].unique(),
                 ticktext = monthly_aov['order_year_month'].unique(),
                 tickangle = -60,
                 showline=False)

fig.update_yaxes(title = 'Orders', showgrid=False, secondary_y=False)
fig.update_yaxes(title = "Sales in ($R)", showgrid=False, secondary_y=True)

fig.add_trace(go.Scatter(
    x = monthly_aov['order_year_month'],
    y = monthly_aov['sales'],
    name = "Sales",
    mode="lines+markers",
    line = dict(color= "#4ECB8D"),
    marker=dict(color=line_colors),
    hovertemplate = 'Sales: %{y:.2s}<extra></extra>'
    ),secondary_y=True)


fig.update_layout(title="<span style='font-size:16px'>Monthly Sales and Orders - 2017 to 2018</span><br><span style='font-size:14px'>(with forecast for Q4 2018)</span>",
                  template="plotly_dark",
                  height=600,
                  hovermode='x unified')

fig.show()


In [ ]:
customer_orders

In [ ]:
monthly_aov.loc[monthly_aov.query("type == 'Actual'")['sales'].idxmax()]["order_year_month"]

<h3>Strong seasonal spikes in sales do not correspond to high AOV</h3>

Sales peak sharply around Nov 2017 → Black Friday then again at Mar–May 2018 and Nov 2018 (forecast), Early 2017 sales are below R$ 400K per month and by late 2018, sales regularly exceed R$ 800K–R$ 1M, This shows strong year-over-year growth and increasing customer demand, yet AOV declines or stays flat during these periods. Peaks in AOV do not always align with peaks in sales. AOV ranges roughly between 125 and 150 BRL.

This indicates high sales are volume-driven (more orders), not value-driven. Customers buy more items, but the average basket size remains modest, typical of promotional cycles.

<h3>AOV peaks occur in low-volume months</h3>

Major AOV spikes appear in Jan 2017, Apr 2017 and Sep 2018(forecast), these are usually months when demand is stable or slightly declining thus higher-priced categories (e.g., electronics, home appliances) drive purchases

Customers tend to make big-ticket purchases when not influenced by promotions or seasonal rush. Promotional seasons attract more low-value orders.

However 2018 maintained consistently higher AOV after mid-year. The trendline shows AOV stabilizes around 130–150 BRL which indicates healthier basket composition vs 2017 suggests better product mix, possible improvement in seller offerings


<h3>Product Category Value Matrix: High-Value Niches vs Mass-Volume Categories</h3>

In [ ]:
product_aov = customer_orders.query("order_year > 2016 & order_year_month != '2018-09'").groupby('product_category_name_english').agg(sales=('price','sum'), orders=('order_id', 'nunique')).sort_values(by='orders', ascending=False).reset_index()
product_aov['AOV'] = (product_aov['sales'] / product_aov['orders']).astype(int)

product_aov.sort_values(by='AOV', ascending=False)

linear_avg = product_aov['AOV'].sum() / product_aov['AOV'].count()

fig = px.scatter(product_aov, x="orders", y="AOV", 
                 text= [products if aov > 300 or orders > 1200 else "" for orders, aov, products in zip(product_aov['orders'], product_aov['AOV'], product_aov['product_category_name_english'])],
                 hover_name='product_category_name_english')

fig.add_hline(y=linear_avg, name='Linear(Average)', line_width=2, line_dash="dash", line_color="#4ECB8D", showlegend=True)
fig.update_traces(textposition='middle right', marker=dict(color="#8D4ECB"))
fig.update_xaxes(title="Orders", showgrid=False)
fig.update_yaxes(title="Average Order Value", showgrid=False)

fig.update_layout(title = "Product Category Performance: AOV vs Order Volume", 
                  height = 700, 
                  template="plotly_dark")
fig.show()

<h3>High AOV, Low Volume (Premium Niche Categories)</h3>

Premium categories such as Computers, Small appliances / home oven and coffee, Home appliances 2, Agro_industry_and_commerce have very high AOV (400–1200 BRL+) but low order volume, these categories behave like high-margin, low-frequency purchases. Even with fewer orders, they significantly lift revenue.

<h3>Low AOV, High Volume (Mass-Market Drivers)</h3>

Mass-Market categories namely Bed_bath_table, Health_beauty, Sports_leisure, Computers_accessories, Furniture_decor drives the highest order counts but average AOV stays between 100–200 BRL. These categories sustain the platform’s daily demand and their revenue is volume-driven—not margin-driven.

This aligns with broad appeal, consumer necessity, Lower price points, Heavy competition → price compression.

<h3>Mid-range AOV, Moderate Volume (Core sustainable categories)</h3>

Sustainable categories such as Office_furniture, Musical instruments and Auto sit near or slightly above the average line and deliver balanced performance (neither niche nor mass-market extremes). Hence, these are stable and reliable contributors – not dependency risks and not niche categories.

The inverse relationship between AOV and order volume reflects natural marketplace economics:
<li>High-demand = low-AOV, low-demand = high-AOV.</li>

In [ ]:
states_aov = customer_orders.groupby('customer_state').agg(sales=('price', 'sum'), order=('order_id', 'nunique')).reset_index().sort_values(by='order', ascending=False)
states_aov['aov'] = states_aov['sales'] / states_aov['order']

fig = make_subplots(specs=[[{'secondary_y':True}]])

fig.add_trace(go.Bar(
    x=states_aov['customer_state'],
    y=states_aov['order'],
    name="Orders",
    text=states_aov['order'],
    textposition='outside',
    textfont=dict(size=10),
    marker = dict(color = "#8d4ecb"),
    hovertemplate="Orders:%{y:,}<extra></extra>"
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=states_aov['customer_state'],
    y=states_aov['aov'],
    mode='lines+markers',
    marker = dict(color = "#4ecb8d"),
    name="AOV",
    hovertemplate="AOV:%{y:.0f}<extra></extra>"
),secondary_y=True)

fig.update_xaxes(title="State")
fig.update_yaxes(title="Orders", showgrid=False, secondary_y=False)
fig.update_yaxes(title="Average Order Value", showgrid=False, secondary_y=True)
fig.update_traces(cliponaxis=True)

fig.update_layout(title = "Geographic Performance Analysis: AOV vs Order Volume by State", 
                  height=600,
                  hovermode='x unified', 
                  template="plotly_dark")
fig.show()

<h3>Dominant Markets:</h3>
SP (~41k+ orders) generates massive demand far higher than any other state even without a high AOV (R$126), likewise MG and RJ (~11.5K - ~12.7K) generates lower volume than SP but higher AOV(R$137-143), this indicates large customer base and high frequency purchases revealing a price-sensitive mass market that drags down overall profitability, so with small increase in AOV via upsells and focusing on scaling through marketing and logistics would produce major revenue uplift due to volume.

<h3>Mid Tier States showing Balanced Performance:</h3>
States like RS, PR, SC, BA, DF, ES, GO generates order volumes between (~2k and ~6k) and AOVs mostly in the (R$ 145–170) range. These states are stable and reliable contributors with potential for regional campaigns, targeted promotions, improving customer retention.

<h3>Premium Markets - High AOV Low Volume States:</h3>
Certain states stand out due to high AOV but very low order volume which includes, PB (~R$ 210 AOV) – extremely high, AP (~R$ 198 AOV), RR (~R$ 160–180 AOV), AC (~R$ 195 AOV), But they have only ~50–600 orders. These are high-value customers but niche markets with limited reach or poor logistics coverage, more focus needed on strengthening ad targeting and explore personalized offers as these customers spend significantly more.

<h3>Northern and Remote States Lag in Order Volume:</h3>
States like RO, AM, AP, RR, AC has very low order volume (< 500) but moderate to high AOV values, Low demand is likely due to shipping challenges or limited e-commerce adaption.

# Sellers Performance
<h3>Objective</h3>

To understand how product categories differ in seller density and seller profitability, helping the marketplace identify:

<li>High-competition categories

<li>High-value but under-served categories

<li>Categories with strong growth potential

<li>Low-yield segments that drain marketplace efficiency</li>

In [ ]:
with open('datasets/brazil_states.json', 'r') as file:
    data = json.load(file)

sales_by_state= customer_orders.groupby(["geo_state"])["price"].sum().round().astype(int).reset_index(name="total_sales")

geo_df = gpd.GeoDataFrame.from_features(data["features"]).merge(sales_by_state, left_on="name", right_on="geo_state").set_index("geo_state")

label_positions = {
    "AC": (-70.5, -9.0),
    "AL": (-36.5, -9.7),
    "AM": (-63.5, -4.5),
    "AP": (-52.0, 1.0),
    "BA": (-41.5, -12.5),
    "CE": (-39.0, -5.0),
    "DF": (-47.9, -15.7),
    "ES": (-40.3, -19.8),
    "GO": (-49.5, -16.0),
    "MA": (-44.5, -5.5),
    "MG": (-44.5, -18.5),
    "MS": (-55.0, -20.5),
    "MT": (-55.0, -13.8),
    "PA": (-52.0, -3.0),
    "PB": (-36.0, -7.3),
    "PE": (-37.0, -8.6),
    "PI": (-42.0, -7.0),
    "PR": (-52.5, -25.0),
    "RJ": (-42.5, -22.3),
    "RN": (-36.5, -5.7),
    "RO": (-63.5, -11.0),
    "RR": (-61.0, 1.5),
    "RS": (-53.2, -30.0),
    "SC": (-49.0, -27.0),
    "SE": (-37.3, -10.7),
    "SP": (-47.5, -22.8),
    "TO": (-48.0, -10.0)
}

geo_df["lon"] = geo_df['sigla'].map(lambda x: label_positions[x][0])
geo_df["lat"] = geo_df['sigla'].map(lambda x: label_positions[x][1])

fig = px.choropleth(geo_df,
                   geojson=geo_df.geometry,
                   locations=geo_df.index,
                   color="total_sales",
                   color_continuous_scale='Blues',
                   #hover_name=geo_df.index,
                   #hover_data={'total_sales': True, 'name': True},
                   projection="mercator")

fig.add_scattergeo(
    lon = geo_df['lon'],
    lat = geo_df["lat"],
    text = geo_df.index + "<br>" + geo_df["total_sales"].astype(str),
    mode = "text",
    textfont=dict(color="black", size=11)
)

fig.update_geos(fitbounds="locations", visible=False)
# fig.update_traces(hovertemplate='<b>Region:</b> %{location}<br>' +
#                                 '<b>Value:</b> %{z:$,.0f}<br><extra></extra>')
fig.update_layout(title = 'Geographical Sales Performance: Southeast Dominates Revenue',
                  height=900,
                  margin=dict(l=0, r=0, t=50, b=0),
                  template='plotly_white')
fig.show()

In [ ]:
df_category = customer_orders.groupby('product_category_name_english').agg(total_revenue=('price', 'sum'), sellers=('seller_id', 'nunique')).reset_index()

df_category['avg_per_seller'] = (df_category['total_revenue'] / df_category['sellers']).round()

df_category['bubble_size'] = df_category['total_revenue'] / df_category['total_revenue'].max() * 100

seller_mean = df_category['sellers'].mean()
avg_median = df_category['avg_per_seller'].median()

def classify(row):
    if row['sellers'] < 200 and row['avg_per_seller'] > 5000:
        return "Hidden Gems"
    if row['avg_per_seller'] > 2050:
        return "Rising Stars"
    if row['sellers'] < seller_mean and row['avg_per_seller'] < avg_median:
        return "Low-Yeild Niches"
    else:
        return "Saturation Traps"

df_category['segment'] = df_category.apply(classify, axis=1)

fig = px.scatter(df_category, x="sellers", y="avg_per_seller",
	             size="sellers", color="segment",
                 size_max=60,
                 text= [products if aps > 2500 else "" for aps, products in zip(df_category['avg_per_seller'], df_category['product_category_name_english'])],
                 hover_data=["product_category_name_english"])

fig.update_xaxes(title="Number of Sellers", showgrid=False)
fig.update_yaxes(title="Avg Revenue per Seller", showgrid=False)
fig.update_traces(textposition="middle right", hovertemplate = "%{customdata[0]}<br>Sellers: %{x}<br>Avg Revenue: %{y:.,}", cliponaxis=True)
fig.update_layout(height = 700,
                  title="Marketplace Seller Segmentation: Identifying High-Value and High-Risk Categories",
                  legend = dict(orientation='h', yanchor = 'top', y=1.02, xanchor = 'right', x=1),
                  legend_title='',
                  template="plotly_dark")
fig.show()

<h3>Non Linear Relationship but Inverse:</h3>
The plot clearly shows more sellers ≠ more revenue but shows inverse relationship where many high-seller categories deliver lower avg revenue per seller and some low-seller categories deliver very high avg revenue due to demand dilution, Market saturation reduces revenue per seller, but not uniformly — category demand shapes the curve.

<h3>Neglected Hidden Gems:</h3>
Categories such as computers generate over (R$25000) and watches_gifts (R$12000), and Office Furniture (R$8000) in average stand out as major “Hidden Gems" with exceptionally high revenue compared to others but with very few sellers indicate low competition + strong demand — the best mix for growth., These markets are under-penetrated and represent the highest ROI opportunity for new sellers.


<h3>Rising Stars:</h3>
Mid-sized clusters such as computers_accessories, toys, garden_tools, and cool_stuff shows moderate to high competition (200-300 sellers) with Mid-to-high average revenue per seller (R$3000-6000) these products offers profitable growth potential and these markets have healthy demand and room for differentiation, the remarkable product of this segment is health_beauty which maintains over R$5,000 despite 500+ sellers.

<h3>Miscellaneous Saturation Traps:</h3>
Categories like Furniture Decor, Auto, Housewares, Sports & Leisure show very large number of sellers ranging from (0-500) with low to moderate average revenue per seller (R$1000-2000). These markets are oversaturated, new sellers entering these segments will face intense competition, lower margins, lower revenue per seller as these categories behave like commodity markets. Sellers flood the category because of perceived demand, but only a few succeed hence marketplace should avoid onboarding more sellers here and instead improve product differentiation and promote high-quality or premium alternatives

<h3>Low-Yield Niches</h3>
Categories like fashion_shoes, kitchen, home_appliances, luggage_accessories, audio have few sellers (~60) with very low avg revenue per seller (R$<1000), These categories suffer from weak market demand and limited revenue opportunity for sellers and market entry here brings minimal upside. This segment is not ideal for investment or seller recruitment; should focus instead on cross-selling strategies, bundling and seasonal or targeted campaigns


<h3>Top Sellers: Strength Through Category Focus</h3>
<h4>Objective</h4>

To study how top sellers achieve high revenue — whether they dominate a single high-value category or distribute their sales intelligently across multiple segments.

In [ ]:
test1 = customer_orders.groupby('seller_id')['price'].sum().sort_values(ascending=False).head(10).reset_index()
test1['price'] = test1['price'].astype(str).str.replace(',', '').astype(float)

test2 = customer_orders[customer_orders['seller_id'].isin(test1['seller_id'])][['seller_id', 'price', 'product_category_name_english']]
test3 = test2.merge(df_category[['product_category_name_english', 'segment']], on='product_category_name_english')

test4 = test3.groupby('seller_id')['price'].sum().reset_index().rename(columns={'price':'total_revenue'})
test5 = test3.groupby(['seller_id','segment']).agg(sales = ('price', 'sum'), n_cat = ('product_category_name_english', 'nunique')).reset_index()

test6 = test5.merge(test4, on="seller_id")

test6["share"] = ((test6["sales"] / test6["total_revenue"]) * 100).map(lambda x: f"{x:.2f}")

test6["share_label"] = test6.apply(
    lambda r: f"{r['share']}% ({r['n_cat']} cat)", axis=1
)

pivot_test = test6.pivot_table(
    index="seller_id",
    columns="segment",
    values="share_label",
    aggfunc="first"
).reset_index()

desired_order = ["seller_id", "Hidden Gems", "Rising Stars", "Saturation Traps", "Low-Yeild Niches"]
ordered_cols = [col for col in desired_order if col in pivot_test.columns]

pivot_test = pivot_test[ordered_cols]

# ---- FIX: convert all cells to string for safe processing ----
pivot_test = pivot_test.fillna("0%").astype(str)

def style_color(value):
    # Extract numeric percentage
    num = float(value.split('%')[0])  

    if num == 0:
        return "#F2F2ED"  # dark/no activity
    elif num < 5:
        return "#BDFCC9"
    elif num < 10:
        return "#98FF98"
    elif num < 25:
        return "#39E75F" 
    elif num < 50:
        return "#F4F269" 
    elif num < 75:
        return "#FFBF00"
    elif num < 90:
        return "#F77319"
    else:
        return "#F94449"  # high dominance
    
colors = pivot_test.copy()

for col in pivot_test.columns[1:]:
    colors[col] = pivot_test[col].apply(style_color)

header_colors = "lightgray"
font_color = "black"

fig = go.Figure(data=[go.Table(
    columnwidth=[1.2, 0.5, 0.5, 0.5, 0.5],
    header=dict(
        values=[f"<b>{col}</b>" for col in pivot_test.columns],
        line_color="lightgray",
        fill_color=header_colors,
        font=dict(color=font_color, size=14),
        align="center",
        height=30
    ),
    cells=dict(
        values=[pivot_test[col] for col in pivot_test.columns],
        # FIX: Apply a single static color for the first column, dynamic colors for others
        fill_color=[
            ["#F2F2ED"] * len(pivot_test)  # First column (seller_id)
        ]
        + [
            colors[col] for col in pivot_test.columns[1:]  # Apply dynamic scale to remaining columns
        ],
        line_color="lightgray",
        align="center",
        font=dict(color="black", size=12),
        height=30
    )
)])

fig.update_layout(title="Top Seller Strategy Mapping: Key Segment Dominance vs Multi-Category Strength",
                  height=600, 
                  width = 1200, 
                  template="plotly_dark")
fig.show()



<h3>Top Sellers Tend to Specialize Rather Than Diversify:</h3>
Most high-performing sellers generate the majority of their revenue from one dominant product segment (often above 80–95% share). and avoid high competition saturation traps category where atmost 2 sellers achieve decent success ranging from (66-87% share) and operating across 10 different categories, demonstrating that competing in saturated markets require operational scale.

This suggests that top performers win either by dominating one profitable niche or strategically balancing their portfolio across growth categories while avoiding overexposure to saturated or low-margin segments.

<h3>Hidden Gems Are the Most Profitable Strategic Segment</h3>
Multiple sellers show high revenue concentration in the “Hidden Gems” category (80%–99%), indicating high revenue potential, low competition. Sellers who discover these products can strongly outperform others, Sellers relying on Hidden Gems often pair them with small contributions from Rising Stars, indicating an expansion strategy. These sellers tend to fall into a "Specialized" and “Focused growth” position, scaling a proven high-margin niche. 

The marketplace strategy is to identify these categories and onboard more sellers thus increase product vareity without risking saturation.

<h3>Rising Stars Act as an Expansion Channel, Not a Primary Revenue Driver</h3>
Rising Stars appear mostly at 5–30% contribution levels, they rarely dominate revenue but often accompany a strong core category. This indicates that sellers use Rising Stars for future-proofing and testing emerging demand.

<h3>Saturation Traps Still Matter — But Only for a Few Sellers</h3>
A small subset of top sellers relies heavily (50–80%+ revenue) on Saturation Traps these sellers operate in highly competed product segments where margins risk erosion, This pattern is risky and too much categories may add to pricing pressure or commoditization.

The marketplace strategy is to encourage these sellers to expand into Rising Stars and Hidden Gems to reduce churn risk and improve marketplace profitability.

<h3>Low-Yield Niches Remain Minimal</h3>
No top seller relies on these niches as a primary source — they are support categories, not profit engines. confirming these represent truly low-potential markets that should only serve as supplementary revenue streams. 

The marketplace strategy is to stop pushing these categories for seller acquisition and instead reclassify or merge them.

<h3>Seller Pricing and Category Behaviour Analysis</h3>
To understand how sellers behave across product categories—both in terms of pricing and assortment width—and identify structural differences between one-time sellers and multi-order sellers.

This helps uncover:

<li>Which categories attract opportunistic sellers vs. established sellers

<li>Whether one-time sellers overprice their products

<li>Which categories drive sustained seller activity

<li>Whether diversifying product categories improves seller success</li>

In [ ]:
seller_orders = customer_orders.groupby("seller_id")["order_id"].nunique().reset_index()
seller_orders["seller_type"] = np.where(seller_orders["order_id"] > 1, "multi", "one")

seller_test = customer_orders.merge(seller_orders[["seller_id", "seller_type"]], on="seller_id", how="left")

price_summary = seller_test.groupby(["product_category_name_english", "seller_type"])["price"].mean().reset_index()

avg_one_order_seller = price_summary[price_summary['seller_type'] == 'one'].reset_index(drop=True)

avg_multi = price_summary[price_summary['seller_type'] == 'multi']

avg_multi_order_seller = avg_multi[avg_multi['product_category_name_english'].isin(avg_one_order_seller['product_category_name_english'])].reset_index(drop=True)

fig=go.Figure()
fig.add_trace(go.Scatter(
    x=avg_one_order_seller['product_category_name_english'],
    y=avg_one_order_seller['price'].astype(int),
    mode='lines+markers',
    name="avg_one_order_seller",
    marker = dict(color= "#8d4ecb")
))

fig.add_trace(go.Scatter(
    x=avg_multi_order_seller['product_category_name_english'],
    y=avg_multi_order_seller['price'].astype(int),
    mode='lines+markers',
    name="avg_multi_order_seller",
    marker = dict(color= "#4ecb8d")
))

fig.update_xaxes(title="Product Category", showgrid=False)
fig.update_yaxes(title="Average Price Per Product Category", showgrid=False)

fig.update_layout(height=700, 
                  title="Average Product Price by Category: Price Positioning Differences Between One Time vs Multi-Order Sellers",
                  legend = dict(orientation='h', yanchor = 'top', y=1.10, xanchor = 'right', x=1),
                  legend_title='',
                  template="plotly_dark",
                  hovermode='x unified')
fig.show()


<h3>One-time sellers consistently price higher in most categories</h3>

One-Order Sellers Drastically Overprice in 35 Categories  (nearly 2x more than multi-order sellers) — Mostly in Saturation Traps exactly where the market punishes them the most (electronics, auto, appliances, toys) which is the worst possible place to overprice where competitors dominate. Their pricing spikes cause no repeat orders huge overpricing in categories like computers (5,139 BRL), home appliances (1,065 BRL), art (1,110 BRL) show complete misalignment with real demand. 

<li>This pattern suggests one-time sellers enter the platform with inflated, experimentation, or opportunistic pricing.

<li>Higher prices → lower conversion → sellers make only a single sale → appear as “one-order sellers”. Hence overpricing is a major driver behind one-time sellers not returning.</li>

<h3>Multi-order sellers maintain stable, category-anchored pricing</h3>
Multi-Order Sellers average prices follow a consistent, narrower band,— Strategically in Low-Competition or Value-Adding Niches more in low yield niches and some in rising stars such as (agro_industry_and_commerce, musical_instruments, health_beauty, perfumery) these sellers raise prices only where buyers are value-oriented, not price-sensitive and they are selectively premium never random. They know the competitive price floors and keep prices lower in 80–90% of categories and raise prices only in categories where customers care about quality (books_general_interest, perfumery, music, musical instruments).

<li>Higher sales velocity → more orders → appear as multi-order sellers. Hence pricing discipline is a predictor of sustained seller performance.</li><br>

The Underlying Consumer Behavior Pattern is consumers tolerate high prices when product quality matters (musical instruments, perfumery, health_beauty, books), the product is differentiated (Hidden Gems or Rising Stars) whereas, consumers do not tolerate high prices when the product is standardized, competition is high, electronics are involved.

Seller retention is lost due to fundamental misunderstanding of segment elasticity and price sensitivity, If new sellers were guided on pricing benchmarks per segment, retention would increase immediately.


<h3>Seller Engagement Patterns: Niche vs Diversified Sellers and Their Category Preferences</h3>

In [ ]:
from collections import defaultdict

df_sellers = customer_orders.groupby('seller_id')['product_category_name_english'].nunique().reset_index(name='category')
sellers_category = df_sellers.query('category > 0').groupby('category')['seller_id'].count().reset_index(name='sellers')

category_dict = defaultdict(int)

def group_category(row):
    if row["category"] in [1, 2]:
        category_dict[row["category"]] += row["sellers"]

    if 3 <= row["category"] <= 4:
        category_dict["3-4"] += row["sellers"]

    if 5 <= row["category"] <= 7:
        category_dict["5-7"] += row["sellers"]

    elif row["category"] >= 8:
        category_dict["8+"] += row["sellers"]  

sellers_category.apply(group_category, axis=1)

grouped_category = pd.DataFrame(list(category_dict.items()), columns=['category', 'sellers'])
grouped_category["category"] = grouped_category['category'].astype(str)

#Top 5 product categories by number of one order sellers 
one_order_sellers = seller_test.query("seller_type == 'one'").groupby(["product_category_name_english"])["seller_id"].nunique().nlargest(5).reset_index(name="sellers")

#Top 5 product categories by number of multi order sellers
multi_order_sellers = seller_test.query("seller_type == 'multi'").groupby(["product_category_name_english"])["seller_id"].nunique().nlargest(5).reset_index(name="sellers")

fig = make_subplots(rows=2, cols=2,
                    specs=[[{"type":"xy", "colspan":2}, None],
                           [{"type":"xy"}, {"type":"xy"}]],
                    subplot_titles=["Sellers Behaviour by Category Range: Niche to Diversified", "Top 5 Product Categories among One Order Sellers", "Top 5 Product Categories among Multi Order Sellers"])

fig.add_trace(go.Bar(
    x=grouped_category['category'],
    y=grouped_category['sellers'],
    text=grouped_category['sellers'],
    textposition='outside',
    textfont=dict(size=10),
    marker = dict(color=grouped_category['sellers'], colorscale=["#b389dc", "#8d4ecb", "#672e9e"]),
    showlegend=False,
    hovertemplate="%{x}<br>Sellers:%{y:,}<extra></extra>"
),row=1, col=1)

fig.update_xaxes(title="Number of Product Categories",  row=1, col=1)
fig.update_yaxes(title="Number of Sellers", showgrid=False, row=1, col=1)

fig.add_trace(go.Bar(
    x=one_order_sellers['product_category_name_english'],
    y=one_order_sellers['sellers'],
    text=one_order_sellers['sellers'],
    textposition='outside',
    textfont=dict(size=10),
    marker = dict(color=one_order_sellers['sellers'], colorscale= ["#89dcb3", "#4ecb8d", "#2e9e67"]),
    showlegend=False,
    hovertemplate="%{x}<br>Sellers:%{y:,}<extra></extra>"
),row=2, col=1)

fig.update_xaxes(title="Product Categories",  row=2, col=1)
fig.update_yaxes(title="Number of Sellers", showgrid=False, row=2, col=1)

fig.add_trace(go.Bar(
    x=multi_order_sellers['product_category_name_english'],
    y=multi_order_sellers['sellers'],
    text=multi_order_sellers['sellers'],
    textposition='outside',
    textfont=dict(size=10),
    marker = dict(color=multi_order_sellers["sellers"], colorscale= ["#99e9fe", "#80e4fe", "#01cade"]),
    showlegend=False,
    hovertemplate="%{x}<br>Sellers:%{y:,}<extra></extra>"
),row=2, col=2)

fig.update_xaxes(title="Product Categories",  row=2, col=2)
fig.update_yaxes(title="Number of Sellers", showgrid=False, row=2, col=2)

for ann in fig['layout']['annotations'][:7]:  # Only adjust subplot titles
    ann['y'] += 0.025
fig.update_traces(cliponaxis=False)
fig.update_layout(height=900, 
                  template="plotly_dark")

fig.show()

<h3>Category Range: Niche vs Diversified Sellers</h3> 

<h4>Most sellers operate in a single category (1,728 sellers)</h4>

This indicates that the marketplace is dominated by niche-focused sellers rather than broad catalog sellers. Most sellers enter the market with narrow specialization and involvement declines drastically as complexity rises. This implies that rather than being big shops, the majority of Olist vendors are small, specialized companies.

But niche focus may come with risk such as Limited diversification and Lower cross-category discovery

<h3>The number of sellers declines sharply as category range expands</h3>

The largest obstacle that sellers encounter on the site is the enormous 59% drop-off that occurs when they attempt to expand from one to two categories (1,689→690). With notable decreases ranging from 37% to 57%, each expansion stage is still difficult, but the difficulty fluctuates rather than following a regular pattern.

Only 282 sellers (9%) successfully operate across five or more categories, indicating that most sellers lack the necessary resources or platform expertise.

The marketplace struggles not because of low seller participation, but because of seller misalignment. Most sellers enter with just 1 category and select extremely competitive segments—then overprice their products. Meanwhile, our most successful sellers diversify across 3–20 categories, price correctly, and position themselves in Rising Star categories that reward assortment and trust.
        
To grow marketplace performance, we must guide new sellers toward smarter category choices, diversified portfolios, and competitive pricing strategies. The data makes it clear: success is achieved not by specialization, but by strategic breadth.”

# Temporal Trends

In [ ]:
temp_df = pd.crosstab(customer_orders["order_day_name"], customer_orders["order_hour"])

fig = go.Figure(data=go.Heatmap(z = temp_df.values,
                                x = temp_df.columns,
                                y = temp_df.index,
                                text = temp_df.values.round(2), # Use rounded z values for cleaner text
                                texttemplate = "%{text}", # Display the content of the 'text' property
                                hovertemplate = "%{y}<br>time:%{x}<br>Orders:%{z}<extra></extra>",
                                colorscale = "Viridis"))
fig.update_xaxes(title= "Hours", side="top", tickmode= "array",
                tickvals = customer_orders["order_hour"].unique(),
                ticktext = customer_orders["order_hour"].unique())

day_order = ['Sun', 'Sat', 'Fri', 'Thu', 'Wed', 'Tue', 'Mon']
fig.update_yaxes(title = "Days", tickmode= "array",
                 categoryarray = day_order,
                tickvals = customer_orders["order_day_name"].unique(),
                ticktext = customer_orders["order_day_name"].unique())
fig.update_layout(#title = "Order Volume by Hour and Day of Week",
                  width = 1240, 
                  height = 600, 
                  template="plotly_dark")
fig.show()

# Key Insights:
We can infer from the above heatmap that across every weekday the high demand consistently clusters between 10a.m. to 10p.m. with a sharp rise starting at 9 a.m. and peak intensity usually happens around afternoon 2p.m. to 4p.m. when customers place orders the most frequently especially Mon-Wed, whcih indicates early-week shoppers show high intent as they research over the weekend and buy at the start of the workweek.

Thursday & Friday show gradual drop-off slightly compared to Mon–Wed but still high midday activity peaks around 1p.m.–4p.m. and shows decent traffic until 9p.m. these are still important days, but customers are more selective—better suited for targeted, category-specific promotions rather than broad campaigns.

Saturday & Sunday have lower overall volume but different behavior patterns, During saturday orders grow later from (11a.m. onward) and stays moderate at afternoon levels and are not as strong as weekdays and also the peaks are softer and more spread out. Whereas, sunday also shows the weakest early-morning activity of the entire week but becomes surprisingly active in late afternoon & evening (4p.m.– 10p.m.) which shows weekend shoppers are more casual as the browsing patterns are less structured and sunday evening is an untapped window for “weekend closing deals”.

# Churn Analysis
In this analysis we're going to explore about the churned customers who made one purchase and never came back in the available time window, we'll identify how many customers churned and what set's them apart.

<h3>Objective:</h3>
Identify factors that differentiate Retained customers from Churned customers, using available behavioral, delivery, geographic, and transaction-level features.

⚠️ However, because 98% of customers are classified as churned, many features are heavily skewed. A major part of this analysis includes assessing whether churn insights are statistically meaningful or not.

<h3>Feature Hypotheses</h3>
    Let's explore what are all the features within the scope of our data to identify behavioural contrasts between churned and retained customers

| Feature                | Hypothesis                                                           |
| ---------------------- | -------------------------------------------------------------------- |
| `Total Order Value`    | Churned customers may have made very low or one-time high purchases. |
| `Shipping Delay`       | Delays could push users to avoid repeat purchases.                   |
| `Cohort Month`     | Customer Acquistion by Cohort Month               |



In [ ]:
churn_status = customer_orders.groupby("customer_unique_id")["cohort_index"].agg(lambda x: (x == 0).all()).reset_index(name="is_churned")

customer_churn_data = customer_orders[["order_id", "customer_id", "customer_unique_id", "order_purchase_timestamp", "cohort_month", "product_category_name_english", "price", "delivery_days", "delay_vs_estimate", "customer_state", "geo_state", "review_score"]].merge(churn_status, on="customer_unique_id")

customer_churn_data["cohort_month"] = customer_churn_data["cohort_month"].astype(str)

churned_customers = customer_churn_data.query("is_churned == True").groupby("customer_unique_id").agg({"order_id": "nunique",
                                                                                                       "price" : "sum",
                                                                                                       "delivery_days": "mean",
                                                                                                       "delay_vs_estimate": "mean",
                                                                                                       "cohort_month": "min",
                                                                                                       "customer_state":"first",
                                                                                                       "product_category_name_english": lambda x :  list(x.unique())}).reset_index()

retained_customers = customer_churn_data.query("is_churned == False").groupby("customer_unique_id").agg({"order_id": "nunique",
                                                                                                        "price" : "sum",
                                                                                                        "delivery_days" : "mean",
                                                                                                        "delay_vs_estimate" : "mean",
                                                                                                        "cohort_month":"min",
                                                                                                        "customer_state": "first",
                                                                                                        "product_category_name_english": lambda x : list(x.unique())}).reset_index()



# Behaviour of Churned Vs Retained Customers
    We'll break this into a few focused comaprisons to try and uncover churn and retention patterns. 

1. Churned Vs Retained Summary Comparison Table

In [ ]:
summary = pd.DataFrame({"Metric": ["Customer %", "Average Orders", "Average Spend", "Average delivery days", "Average fast delivery", "Average delivery delay", "Quicker deliveries (%)", "Delayed deliveries (%)"],
                        "Churned": [f"{(len(churned_customers) / len(customer_orders["customer_unique_id"].unique()) * 100):.2f}",
                                    f"{churned_customers["order_id"].mean():.0f}",
                                    f"R$ {churned_customers["price"].mean().round():.0f}",
                                    f"{churned_customers["delivery_days"].mean().round():.0f} days",
                                    f"{churned_customers[churned_customers["delay_vs_estimate"] >= 0]["delay_vs_estimate"].mean().round():.0f} days",
                                    f"{churned_customers[churned_customers["delay_vs_estimate"] < 0]["delay_vs_estimate"].mean().round(0):.0f} days",
                                    (churned_customers[churned_customers["delay_vs_estimate"] >= 0]["delay_vs_estimate"].count() / churned_customers["delay_vs_estimate"].count() * 100).round(2),
                                    (churned_customers[churned_customers["delay_vs_estimate"] < 0]["delay_vs_estimate"].count() / churned_customers["delay_vs_estimate"].count() * 100).round(2)],

                        "Retained": [f"{(len(retained_customers) / len(customer_orders["customer_unique_id"].unique()) * 100):.2f}",
                                    f"{retained_customers["order_id"].mean():.0f}",
                                    f"R$ {retained_customers["price"].mean().round():.0f}",
                                    f"{retained_customers["delivery_days"].mean().round():.0f} days",
                                    f"{retained_customers[retained_customers["delay_vs_estimate"] >= 0]["delay_vs_estimate"].mean().round():.0f} days",
                                    f"{retained_customers[retained_customers["delay_vs_estimate"] < 0]["delay_vs_estimate"].mean().round(0):.0f} days",
                                    (retained_customers[retained_customers["delay_vs_estimate"] >= 0]["delay_vs_estimate"].count() / retained_customers["delay_vs_estimate"].count() * 100).round(2),
                                    (retained_customers[retained_customers["delay_vs_estimate"] < 0]["delay_vs_estimate"].count() / retained_customers["delay_vs_estimate"].count() * 100).round(2)]})

headerColor = 'grey'
rowEvenColor = '#4ECB8D'
rowOddColor = '#8D4ECB'

#Table
fig = go.Figure(data=(go.Table(
  columnwidth=[0.8, 0.8, 0.8],
  header=dict(
    values=list(summary.columns),
    line_color='darkslategray',
    fill_color=headerColor,
    align=['left','center'],
    font=dict(color='white', size=15)
  ),
  cells=dict(
    values=[summary[col].to_list() for col in summary.columns],
    line_color='darkslategray',
    # 2-D list of colors for alternating rows
    fill_color = [[rowEvenColor,rowOddColor, rowEvenColor,rowOddColor,rowEvenColor,rowOddColor, rowEvenColor, rowOddColor]*8],
    align = ['left', 'center'],
    height = 27,
    font = dict(color = 'white', size = 14)
    ))))

fig.update_layout(template="plotly_dark")
fig.show()

“This e-commerce marketplace has an extremely high proportion of one-time buyers (~98%), which is typical for non-subscription retail. Therefore, churn reflects the inherent buying pattern rather than customer dissatisfaction. As a result, churn analysis should be interpreted carefully, focusing on behavioral contrasts rather than causation.”

Because churned users dominate the dataset (98%), the average metrics for the churned segment represent the platform’s typical customer, not necessarily a dissatisfied one.
Retained customers are the exception, and their metrics reveal the profile of a “high-value survivor” with higher spend, more orders and lower dissatisfaction markers

This means:
➡️ The question isn’t “Why did churned customers leave?” — it’s “What made the 1.8% stay?”

The “retained” group represents <2% of customers and behaves like high-value “Champions” segment in RFM analysis.


1. Compare Order Value (Churned vs Retained)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=churned_customers["order_id"], name='Churned', marker_color = '#8D4ECB'))
fig.add_trace(go.Histogram(x=retained_customers["order_id"], name='Retained', marker_color =  '#4ECB8D'))

fig.update_xaxes(title="Number of Orders")
fig.update_yaxes(title="Total Counts", showgrid=False)
fig.update_layout(#barmode='overlay', 
                  template="plotly_dark",
                  hovermode='x unified')
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.show()

<h3>Order Value Comparison (Churned vs Retained)</h3>
The distribution reveals

<li>Churned customers are overwhelmingly one-time buyers, with almost all making only 1 order.

<li>Retained customers show multi-order behavior, extending up to 16 orders.</li>

This distribution is not “churn behavior” but natural buyer lifecycle behavior in online marketplaces.

In markets like Brazil’s e-commerce: Many customers shop once, often during discounts or specific needs, Repeat customers are rare but far more profitable.

<h3>2. Shipping Delay</h3>

In [ ]:
x1 = churned_customers["delay_vs_estimate"].dropna()
x2 = retained_customers["delay_vs_estimate"].dropna()

hist_data = [x1, x2]
group_labels = ["Churned", "Retained"]

fig = ff.create_distplot(hist_data, group_labels, bin_size=.2, show_hist=False, show_rug=True, show_curve=True)
fig.update_layout(xaxis_title="Late vs Early delivery", template="plotly_dark")
fig.show()

<h3>Shipping Speed vs Delivery Delay</h3>
the delay distribution shows

<li>Both groups cluster around deliveries happening close to expected date.

<li>Churned customers show a wider spread, including extreme late deliveries (up to +150 days).

<li>Retained customers almost never experience these extreme delays.</li>

Delivery delay might contribute to churn — but only for a tiny fraction. Since 98% are churned, late delivery cannot explain churn, because nearly all churned customers still had “normal” delivery timelines.

Instead, the insight is:
<li> Extreme late deliveries limit the chance of a customer ever returning
<li> But on-time delivery does not guarantee retention</li>

This aligns with global e-commerce patterns:

➡️ Good delivery keeps customers neutral

➡️ Bad delivery destroys loyalty

<h3>3. Cohort Month Churn Trend</h3>

In [ ]:
excluded_values = ['2016-09', '2016-10', '2016-12']
cohort_churn = churned_customers[~churned_customers["cohort_month"].isin(excluded_values)]["cohort_month"].value_counts().reset_index().sort_values(by = "cohort_month")
cohort_retain = retained_customers[~retained_customers["cohort_month"].isin(excluded_values)]["cohort_month"].value_counts().reset_index().sort_values(by = "cohort_month")

fig = go.Figure()
fig.add_trace(go.Scatter(mode = "lines+markers",
                        x = cohort_churn["cohort_month"],
                        y = cohort_churn["count"],
                        name='Churned',
                        marker_color='#8d4ecb'))

fig.add_trace(go.Scatter(mode = "lines+markers",
                        x = cohort_churn["cohort_month"],
                        y = cohort_retain["count"],
                        name='Retained',
                        marker_color='#43cb8d'))
fig.update_xaxes(title = "Cohort Month", tickmode = "array", 
                tickvals = cohort_churn["cohort_month"],
                ticktext = cohort_churn["cohort_month"],
                tickangle = -60,
                showgrid=False)
fig.update_yaxes(title = "Number of Customers", showgrid=False)
fig.update_layout(title = "Customer Acquisition by Cohort Month", height = 700, hovermode = 'x unified', template="plotly_dark")
fig.show()